# Exact Drifting Model — Fully Retuned Geodesic-Loss Experiment

This is a standalone **Exact-only** dissertation experiment.

Unlike the earlier geodesic-loss ablation, this notebook does **not** reuse the
ambient-loss Exact bandwidth. It performs a fresh, validation-only hyperparameter
selection using the geodesic fixed-point loss itself, then trains the selected
configuration for 50,000 steps with the same cosine learning-rate schedule used in
the final V3/V4 experiments.

### Protocol

**Tuning**
- Exact model only.
- Loss: squared geodesic distance to the frozen transported target.
- Grid: \(c\in\{1/32,1/16,1/8,1/4,1/2,1\}\).
- Seeds: 43, 44, 45.
- Maximum 15,000 steps per candidate.
- Same original early-stopping rule.
- Original tuning LR schedule: \(10^{-3}\) through step 5,000, cosine to \(10^{-5}\) at step 15,000.
- Selection criterion: mean validation **raw ambient Gaussian MMD²** across the three tuning seeds.

**Final fitting**
- Uses the newly selected geodesic-loss bandwidth.
- Seeds: 101, 102, 103, 104, 105.
- Exactly 50,000 optimisation steps; no early stopping.
- LR \(10^{-3}\) through step 5,000, then cosine annealed to \(10^{-5}\) at step 50,000.
- Best checkpoint selected by validation raw ambient Gaussian MMD².
- Test data are evaluated only after the validation-selected checkpoint is frozen.

All outputs are written to a new directory and do not overwrite the original Exact runs.


## 1. Imports, paths, run controls, and the locked protocol

The notebook is designed for Google Colab + Drive, but can also run locally by setting `DISSERTATION_DRIVE_ROOT`.

The defaults run all 15 targets. For practical scheduling, `ACTIVE_DATASETS` and `ACTIVE_METHODS` can restrict a session without changing the experiment definition. Completed runs are cached atomically and reused when `RESUME=True`.

In [ ]:
import os
import sys
import math
import json
import copy
import hashlib
import random
from pathlib import Path
from dataclasses import dataclass, asdict, field
from abc import ABC, abstractmethod
from typing import Callable, Dict, List, Optional, Sequence, Tuple, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path, connected_components

IN_COLAB = (
    'google.colab' in sys.modules
    or os.environ.get('COLAB_GPU') is not None
    or os.environ.get('COLAB_TPU_ADDR') is not None
)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif (not IN_COLAB) and hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path(os.environ.get('DISSERTATION_DRIVE_ROOT', Path.cwd()))

EARTH_DATA_DIR = DRIVE_ROOT / 'earth_dataset'
PROTEIN_MASTER_PATH = DRIVE_ROOT / 'protein_dataset' / 'top500_angles_master.tsv'
AMASS_DIR = DRIVE_ROOT / 'AMASS_dataset'
DISSERTATION_ROOT = DRIVE_ROOT / 'dissertation'
RESULTS_ROOT = DISSERTATION_ROOT / 'final_four_model_all_manifolds_v1'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# Prepared AMASS READY files. Primary target = absolute root orientation.
SO3_DATASET_FILES = {
    'BMLhandball': AMASS_DIR / 'BMLhandball_SO3_ready_10hz.npz',
    'DanceDB': AMASS_DIR / 'DanceDB_SO3_ready_10hz.npz',
    'BMLmovi': AMASS_DIR / 'BMLmovi_SO3_ready_10hz.npz',
    'CMU': AMASS_DIR / 'CMU_SO3_ready_10hz.npz',
}
SO3_REPRESENTATION = 'abs'

# Family switches.
RUN_SPHERE = True
RUN_TORUS = True
RUN_SO3 = True

# Session scheduling controls. None = all available datasets/methods.
ACTIVE_DATASETS = None
ACTIVE_METHODS = None  # subset of {'exact','partial','full','euclidean'}

RESUME = True
FORCE_RERUN = False
SHOW_FINAL_PLOTS = True
SHOW_TUNING_PLOTS = True
SHOW_TRAINING_DIAGNOSTICS = False

EXPERIMENT_FORMAT_VERSION = 6

print(f'Device: {DEVICE} (IN_COLAB={IN_COLAB})')
print('Results root:', RESULTS_ROOT)

Mounted at /content/drive
Device: cuda (IN_COLAB=True)
Results root: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1


In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    max_steps: int = 15_000
    learning_rate: float = 1e-3
    min_learning_rate: float = 1e-5
    lr_decay_start_step: int = 5_000

    eta: float = 1.0
    latent_dim: int = 6
    hidden_dim: int = 256

    ema_decay: float = 0.999
    gradient_clip_norm: float = 1.0

    validate_every: int = 500
    early_stopping_start_step: int = 5_000
    early_stopping_patience: int = 10
    min_delta_rel: float = 0.005
    min_delta_abs: float = 0.0

    history_every: int = 100

    # Graph distance is the computational bottleneck. Every method on a
    # dataset uses the SAME effective batch size, capped here for fairness.
    common_batch_cap: int = 256

    # Common safeguard applied in the units of each method's displacement.
    max_step: float = 0.25


@dataclass(frozen=True)
class ExperimentConfig:
    tuning_seeds: Tuple[int, ...] = (43, 44, 45)
    final_seeds: Tuple[int, ...] = (101, 102, 103, 104, 105)

    validation_noise_seeds: Tuple[int, ...] = (1001, 1002, 1003)
    test_noise_seeds: Tuple[int, ...] = (2001, 2002, 2003)
    validation_subset_seed: int = 4001
    test_subset_seed: int = 4002

    median_pair_seed: int = 3001
    median_num_pairs: int = 100_000

    validation_eval_max: int = 2_048
    test_eval_max: int = 2_048

    c_grid: Tuple[float, ...] = (1/32, 1/16, 1/8, 1/4, 1/2, 1.0)
    k_graph_grid: Tuple[int, ...] = (4, 8, 16, 32, 64)

    # Fixed approximate-geometry construction.
    geometry_max_points: int = 2_500
    dedup_tol: float = 1e-7
    fps_candidate_max: int = 6_000
    graph_mode: str = 'mutual'
    graph_num_anchors: int = 4
    projection_num_charts: int = 4
    projection_iterations: int = 1

    canonical_seed: int = 101
    real_vs_real_repeats: int = 20


TRAINING_CONFIG = TrainingConfig()
EXPERIMENT_CONFIG = ExperimentConfig()

METHOD_LABELS = {
    'exact': 'Exact',
    'partial': 'Partial (graph)',
    'full': 'Full (graph)',
    'euclidean': 'Euclidean',
}

print(TRAINING_CONFIG)
print(EXPERIMENT_CONFIG)

TrainingConfig(max_steps=15000, learning_rate=0.001, min_learning_rate=1e-05, lr_decay_start_step=5000, eta=1.0, latent_dim=6, hidden_dim=256, ema_decay=0.999, gradient_clip_norm=1.0, validate_every=500, early_stopping_start_step=5000, early_stopping_patience=10, min_delta_rel=0.005, min_delta_abs=0.0, history_every=100, common_batch_cap=256, max_step=0.25)
ExperimentConfig(tuning_seeds=(43, 44, 45), final_seeds=(101, 102, 103, 104, 105), validation_noise_seeds=(1001, 1002, 1003), test_noise_seeds=(2001, 2002, 2003), validation_subset_seed=4001, test_subset_seed=4002, median_pair_seed=3001, median_num_pairs=100000, validation_eval_max=2048, test_eval_max=2048, c_grid=(0.03125, 0.0625, 0.125, 0.25, 0.5, 1.0), k_graph_grid=(4, 8, 16, 32, 64), geometry_max_points=2500, dedup_tol=1e-07, fps_candidate_max=6000, graph_mode='mutual', graph_num_anchors=4, projection_num_charts=4, projection_iterations=1, canonical_seed=101, real_vs_real_repeats=20)


### Fixed design choices for this Exact-geodesic experiment

This notebook reuses the same datasets, architecture, optimiser, EMA, evaluation objects,
drift construction, step cap, gradient clipping, validation metric, tuning seeds, and
hyperparameter grid as the locked four-model experiment.

The deliberate change is that **Exact is trained with the intrinsic fixed-point loss**
\[
    \mathcal L_{\mathrm{geo}}
    =
    \frac{1}{B}\sum_i
    d_{\mathcal M}\!\left(x_i,\operatorname{sg}(\widetilde x_i)\right)^2.
\]

Crucially, this loss is used **both during hyperparameter tuning and during final fitting**.
Therefore the selected bandwidth is genuinely selected for the geodesic-loss Exact model,
rather than inherited from the ambient-loss Exact model.

The tuning protocol itself is unchanged: the full
\(c\in\{1/32,1/16,1/8,1/4,1/2,1\}\) grid, three tuning seeds
\(43,44,45\), a 15,000-step maximum budget with the original early-stopping rule,
and selection by mean validation **raw ambient Gaussian MMD²**.

Final fitting uses five fresh seeds \(101,\ldots,105\), 50,000 optimisation steps,
no early stopping, learning rate \(10^{-3}\) through step 5,000, then cosine annealing
to \(10^{-5}\) at step 50,000. The best final checkpoint is still selected only by
validation raw ambient MMD²; the frozen test set is evaluated only after checkpoint
selection.


## 2. Data-loading helpers

In [ ]:
def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}\n"
            "Edit the path settings near the top of the notebook."
        )
    return path


def load_and_prepare_earth_data(csv_path: Path, device=DEVICE):
    csv_path = require_file(Path(csv_path))
    df = pd.read_csv(
        csv_path,
        header=None,
        names=["latitude", "longitude"],
    )

    latitude_rad = np.deg2rad(df["latitude"].values)
    longitude_rad = np.deg2rad(df["longitude"].values)

    x = np.cos(latitude_rad) * np.cos(longitude_rad)
    y = np.cos(latitude_rad) * np.sin(longitude_rad)
    z = np.sin(latitude_rad)

    coords = torch.tensor(
        np.stack([x, y, z], axis=1),
        dtype=torch.float32,
        device=device,
    )
    return coords


In [ ]:
def split_dataset(
    data: torch.Tensor,
    train_fraction: float = 0.8,
    val_fraction: float = 0.1,
    seed: int = 42,
):
    if not 0 < train_fraction < 1:
        raise ValueError("train_fraction must be between 0 and 1.")
    if not 0 <= val_fraction < 1:
        raise ValueError("val_fraction must be between 0 and 1.")
    if train_fraction + val_fraction >= 1:
        raise ValueError("train_fraction + val_fraction must be less than 1.")

    n = data.shape[0]
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)
    permutation = torch.randperm(n, generator=generator)

    n_train = int(train_fraction * n)
    n_val = int(val_fraction * n)

    train_idx = permutation[:n_train]
    val_idx = permutation[n_train:n_train + n_val]
    test_idx = permutation[n_train + n_val:]

    return (
        data.index_select(0, train_idx.to(data.device)),
        data.index_select(0, val_idx.to(data.device)),
        data.index_select(0, test_idx.to(data.device)),
    )


def sample_from_dataset(dataset, n, generator=None, device=DEVICE):
    if n <= 0:
        raise ValueError("n must be positive.")
    indices = torch.randint(
        low=0,
        high=dataset.shape[0],
        size=(n,),
        generator=generator,
        device="cpu",
    )
    return dataset.index_select(
        0, indices.to(dataset.device)
    ).to(device)


def make_dataset_sampler(dataset, device=DEVICE):
    def sampler(n, generator=None):
        return sample_from_dataset(
            dataset=dataset,
            n=n,
            generator=generator,
            device=device,
        )
    return sampler


def fixed_dataset_subset(dataset, n, seed, device=DEVICE):
    n = min(int(n), len(dataset))
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)
    indices = torch.randperm(
        len(dataset),
        generator=generator,
    )[:n]
    return dataset.index_select(
        0, indices.to(dataset.device)
    ).to(device)


def sample_target_with_seed(sampler, n, seed, device=DEVICE):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)
    return sampler(n, generator=generator).to(device)


def dataframe_to_torus(df, torus, device=DEVICE):
    angles_deg = torch.tensor(
        df[["phi", "psi"]].to_numpy(dtype=np.float32),
        device=device,
    )
    return torus.from_angles(torch.deg2rad(angles_deg))

In [ ]:
def load_amass_ready(path, representation='abs'):
    """Load a prepared AMASS SO(3) READY file and verify the split metadata."""
    path = require_file(Path(path))
    with np.load(path, allow_pickle=True) as d:
        required = [f'train_{representation}', f'val_{representation}', f'test_{representation}']
        for key in required:
            if key not in d:
                raise KeyError(f'{path.name} is missing {key}')

        train = torch.from_numpy(np.asarray(d[f'train_{representation}'], dtype=np.float32))
        val = torch.from_numpy(np.asarray(d[f'val_{representation}'], dtype=np.float32))
        test = torch.from_numpy(np.asarray(d[f'test_{representation}'], dtype=np.float32))

        seq = {}
        for split in ('train', 'val', 'test'):
            key = f'{split}_sequence'
            seq[split] = set(map(str, d[key].tolist())) if key in d else set()

        split_note = str(d['split_note'].item()) if 'split_note' in d else ''
        representation_note = str(d['representation_note'].item()) if 'representation_note' in d else ''

    for name, arr in [('train', train), ('val', val), ('test', test)]:
        if arr.ndim != 2 or arr.shape[1] != 9:
            raise ValueError(f'{path.name}/{name}: expected shape (N,9), got {tuple(arr.shape)}')
        if not torch.isfinite(arr).all():
            raise ValueError(f'{path.name}/{name}: non-finite values detected')

    # If sequence IDs are stored, enforce whole-sequence split disjointness.
    if all(len(seq[s]) > 0 for s in ('train','val','test')):
        assert seq['train'].isdisjoint(seq['val'])
        assert seq['train'].isdisjoint(seq['test'])
        assert seq['val'].isdisjoint(seq['test'])

    return {
        'train': train,
        'val': val,
        'test': test,
        'sequence_sets': seq,
        'split_note': split_note,
        'representation_note': representation_note,
        'path': str(path),
    }

## 3. Exact manifold geometries

In [ ]:
class Manifold(ABC):
  def __init__(self, ambient_dim: int):
    self.ambient_dim = ambient_dim

  @abstractmethod
  def project(self, x):
    """
    Project an ambient point onto the manifold.

    Parameters
    ----------
    x : Tensor
        Point in the ambient space.

    Returns
    -------
    Tensor
        Closest point on the manifold.
    """
    pass

  @abstractmethod
  def project_tangent(self, x, v):
    """
    Project an ambient vector onto the tangent space at x.

    Parameters
    ----------
    x : Tensor
    Point on the manifold.

    v : Tensor
    Ambient vector.

    Returns
    -------
    Tensor
    Tangent vector in T_xM.
    """
    pass

  @abstractmethod
  def exp(self,x,v):
    """
    x: point on manifold
    v: tangent vector

    exponential map returns the point on manifold reached by following
    the unique geodesic starting at x whose initial velocity is v for a distance equal to
    the norm ∥v∥
    """
    pass

  @abstractmethod
  def log(self,x,y):
    """
    x: point on manifold
    y: point on manifold

    logarithmic map returns the initial velocity of the geodesic starting at x and reaching y
    """
    pass

  @abstractmethod
  def dist(self,x,y):
    """
    Geodesic distance between x and y.
    Returns
    -------
    float or Tensor
    Distance along the manifold.
    """
    pass

In [ ]:
class Sphere(Manifold):

    def __init__(self, ambient_dim=3, atol=1e-6):
        if ambient_dim < 3:
            raise ValueError("Sphere manifold requires at least 3 dimensions.")

        super().__init__(ambient_dim)

        self.manifold_dim = ambient_dim - 1
        self.atol = atol
        self.injectivity_radius = math.pi

    # helper functions

    def _check_last_dim(self, x):
        if x.shape[-1] != self.ambient_dim:
            raise ValueError(f"Expected last dimension to be {self.ambient_dim}, but got {x.shape[-1]}.")

    def _point_norm(self, x):
        return torch.linalg.norm(x, dim=-1,)

    def _inner(self, x, y):
        return torch.sum(x * y, dim=-1,)

    # validation functions

    def _check_point(self, x):
        self._check_last_dim(x)

        norms = self._point_norm(x)

        if not torch.isfinite(norms).all():
            raise ValueError("Point norms contain non-finite values.")

        # |a-b| < atol + rtol|b|
        if not torch.allclose(norms, torch.ones_like(norms),
                              atol=self.atol, rtol=0.0,):

            max_err = torch.max(torch.abs(norms - 1.0)).item()
            raise ValueError(f"Points are not on the unit sphere. Maximum norm error = {max_err:.3e}")

        return True

    def _check_tangent(self, x, v):
        self._check_last_dim(x)
        self._check_last_dim(v)
        self._check_point(x)

        x, v = torch.broadcast_tensors(x, v) # enable them to broadcast, always add 1 to the left

        # suppose x (5,3)/ v (3,)
        # then v turned into (1,3), so calculate inner with every row of x

        inner = self._inner(x, v)

        if not torch.isfinite(inner).all():

            raise ValueError("Tangent-space inner products contain non-finite values.")

        if not torch.allclose(inner, torch.zeros_like(inner), atol=self.atol, rtol=0.0,):

            max_err = torch.max(torch.abs(inner)).item()
            raise ValueError(f"v is not in the tangent space at x. Maximum inner product = {max_err:.3e}")

        return True

    # geometric operations

    def project(self, x):
        self._check_last_dim(x)

        norm = self._point_norm(x).unsqueeze(-1) # (n,1)

        eps = 1e-12
        safe_norm = norm.clamp_min(eps) # ensures division numerically safe

        projected = x / safe_norm

        # A zero ambient vector has no direction. Use the first coordinate axis as a deterministic fallback point
        fallback = torch.zeros_like(x)
        fallback[..., 0] = 1.0 # makes first entry 1, else 0: (1,0,0...)

        return torch.where(norm > eps, projected, fallback,)

    def project_tangent(self, x, v):
        self._check_last_dim(x)
        self._check_last_dim(v)
        self._check_point(x)

        x, v = torch.broadcast_tensors(x, v)

        inner = self._inner(x, v).unsqueeze(-1)

        return v - inner * x

    def exp(self, x, v):
        self._check_point(x)
        self._check_tangent(x, v)

        x, v = torch.broadcast_tensors(x, v)

        v_norm = torch.linalg.norm(v, dim=-1, keepdim=True,)

        eps = 1e-12

        safe_v_norm = torch.where(v_norm > eps, v_norm, torch.ones_like(v_norm),)

        sinc = (torch.sin(safe_v_norm)/ safe_v_norm)

        # Taylor approximation near zero:
        # sin(r) / r = 1 - r^2 / 6 + O(r^4)
        sinc = torch.where(v_norm > eps, sinc, 1.0 - (v_norm ** 2) / 6.0,)

        out = (torch.cos(v_norm) * x + sinc * v)

        # projection removes small floating-point errors.
        return self.project(out)

    def is_antipodal(self, x, y):
        """
        Identify pairs at the cut locus of x.

        On the unit sphere, the cut locus of x is the single
        antipodal point -x.
        """
        angle = self.dist(x, y)

        return (self.injectivity_radius - angle <= self.atol)

    def log(self, x, y):
        """
        Principal spherical logarithmic map away from the antipode.

        At exact antipodal pairs the logarithmic map is not unique.
        The returned zero vector is a safe placeholder;
        compute_drift_manifold explicitly assigns zero kernel
        weight to those undefined pairwise interactions.
        """
        self._check_point(x)
        self._check_point(y)

        x, y = torch.broadcast_tensors(x, y)

        angle = self.dist(x, y).unsqueeze(-1)

        dot = self._inner(x, y).unsqueeze(-1)
        tangent_direction = y - dot * x

        direction_norm = torch.linalg.norm(tangent_direction, dim=-1, keepdim=True,)

        eps = 1e-12
        safe_direction_norm = direction_norm.clamp_min(eps)

        out = (angle * tangent_direction / safe_direction_norm)

        is_same_point = (angle <= self.atol)

        is_antipodal = (self.is_antipodal(x, y).unsqueeze(-1) & (direction_norm <= eps))

        # log_x(x) is genuinely zero.
        out = torch.where(is_same_point, torch.zeros_like(out), out,)

        # log_x(-x) is not unique. This value is only a safe
        # placeholder; the corresponding drift weight is masked.
        out = torch.where(is_antipodal, torch.zeros_like(out), out,)

        return self.project_tangent(x, out)

    def dist(self, x, y):
        """
        Stable geodesic distance on the unit sphere:

            d(x, y) = 2 atan2(||x - y||, ||x + y||).

        This gives exactly zero at identical points and pi at
        antipodal points, up to floating-point precision.
        """
        self._check_point(x)
        self._check_point(y)

        x, y = torch.broadcast_tensors(x, y)

        diff_norm = torch.linalg.norm(x - y, dim=-1,)

        sum_norm = torch.linalg.norm(x + y, dim=-1,)

        return 2.0 * torch.atan2(diff_norm, sum_norm,)

    def sample(self, n, device=None, dtype=None, generator=None,):
        x = torch.randn(n, self.ambient_dim, device=device, dtype=dtype, generator=generator, )

        return self.project(x)


sphere = Sphere(ambient_dim=3)

In [ ]:
class FlatTorus(Manifold):
    """
    Flat n-dimensional torus:
    T^n = S^1 * ... * S^1

    Each circle is represented extrinsically in R^2, so:
    manifold dimension = n_circles
    ambient dimension = 2 * n_circles

    For the protein/checkerboard experiment:
    n_circles = 2
    T^2 is represented in R^4.
    """
    def __init__(self, n_circles=2, atol=1e-6):

        if n_circles < 1:
            raise ValueError("Flat torus requires at least one circle.")

        self.n_circles = n_circles

        # each circle S^1 is embedded in R^2
        ambient_dim = 2 * n_circles
        super().__init__(ambient_dim)

        self.manifold_dim = n_circles
        self.atol = atol

        self.injectivity_radius = math.pi

    # helper functions

    def _check_last_dim(self, x):
        if x.shape[-1] != self.ambient_dim:
            raise ValueError(f"Expected last dimension to be {self.ambient_dim}, but got {x.shape[-1]}.")

    def _reshape_blocks(self, x):
        """
        Convert (..., 2 * n_circles)

        into (..., n_circles, 2).

        For T^2: (..., 4) -> (..., 2, 2)
        """
        self._check_last_dim(x)

        return x.reshape(*x.shape[:-1], # unpacks for batch size etc, only changes last dim
                         self.n_circles, # always 2
                         2,)
        # suppose batch 6 of 4
        # changes to 6 of 2x2

    def _block_norm(self, x):
        """
        Return the norm of each R^2 circle block.

        Input: (..., 2 * n_circles)

        Output: (..., n_circles)
        """
        x_blocks = self._reshape_blocks(x)

        return torch.linalg.norm(x_blocks, dim=-1,)

    def _block_inner(self, x, y):
        """
        Compute inner products separately for each circle block.

        Broadcasting is allowed.
        """
        x, y = torch.broadcast_tensors(x, y) # broadcast shapes of x&y

        x_blocks = self._reshape_blocks(x)
        y_blocks = self._reshape_blocks(y)

        return torch.sum(x_blocks * y_blocks, dim=-1,)
        # computes x11*y11, x12*y12 etc
        # then sum x11*y11 + x12*y12

    # Validation functions

    def _check_point(self, x):
        """
        Check that every R^2 block has unit norm.
        """
        self._check_last_dim(x)

        norms = self._block_norm(x)

        if torch.isnan(norms).any():
            raise ValueError("Point block norms contain NaN values.")

        if not torch.allclose(norms, torch.ones_like(norms), atol=self.atol, rtol=0.0,):
            max_err = torch.max(
                torch.abs(norms - 1.0)
            ).item()

            raise ValueError(
                "Points are not on the flat torus. "
                f"Maximum block-norm error = {max_err:.3e}"
            )

        return True

    def _check_tangent(self, x, v):
        """
        For every circle block j, verify
         <x_j, v_j> = 0.

        x and v may be broadcastable rather than having
        exactly the same shape.
        """
        self._check_last_dim(x)
        self._check_last_dim(v)
        self._check_point(x)

        x, v = torch.broadcast_tensors(x, v)

        inner = self._block_inner(x, v)

        if torch.isnan(inner).any():
            raise ValueError(
                "Tangent-space inner products contain NaNs."
            )

        if not torch.allclose(
            inner, torch.zeros_like(inner),
            atol=self.atol, rtol=0.0,
        ):
            max_err = torch.max(
                torch.abs(inner)
            ).item()

            raise ValueError(
                "v is not tangent to every circle factor at x. "
                f"Maximum block inner product = {max_err:.3e}"
            )

        return True

    # geometric operations

    def project(self, x):
        """
        Project each R^2 block onto its unit circle.

        For T^2: (a, b, c, d)

        becomes
        ((a, b) / ||(a, b)||,
        (c, d) / ||(c, d)||).
        """
        self._check_last_dim(x)

        x_blocks = self._reshape_blocks(x)

        norms = torch.linalg.norm(
            x_blocks,
            dim=-1,
            keepdim=True,
        )

        eps = 1e-12
        safe_norms = norms.clamp_min(eps)

        projected = x_blocks / safe_norms

        # if a network outputs an exactly zero block, its
        # direction is undefined. Use (1, 0) as a safe fallback.
        fallback = torch.zeros_like(x_blocks)
        fallback[..., 0] = 1.0 # ... means all preceding dimensions

        projected = torch.where(
            norms > eps, projected, fallback,
        )

        return projected.reshape(
            *x.shape[:-1], self.ambient_dim,
        )
        # conversts back shape from 2,2 to 4

    def project_tangent(self, x, v):
        """
        Project each R^2 block of v onto the tangent line
        of its corresponding circle at x.
        """
        self._check_last_dim(x)
        self._check_last_dim(v)
        self._check_point(x)

        # permits pairwise broadcasted calculations such as:
        # x: (G, 1, 4)
        # v: (G, P, 4)
        x, v = torch.broadcast_tensors(x, v)

        x_blocks = self._reshape_blocks(x)
        v_blocks = self._reshape_blocks(v)

        inner = torch.sum(
            x_blocks * v_blocks,
            dim=-1, keepdim=True,
        )

        projected = (
            v_blocks - inner * x_blocks
        )

        return projected.reshape(
            *projected.shape[:-2],
            self.ambient_dim,
        )

    def exp(self, x, v):
        """
        Product exponential map.

        Apply the circle exponential map independently
        to every circle factor.
        """
        self._check_point(x)
        self._check_tangent(x, v)

        x, v = torch.broadcast_tensors(x, v)

        x_blocks = self._reshape_blocks(x)
        v_blocks = self._reshape_blocks(v)

        # one norm for each tangent vector block
        v_norm = torch.linalg.norm(
            v_blocks,
            dim=-1, keepdim=True,
        )

        eps = 1e-12

        safe_v_norm = torch.where(
            v_norm > eps,
            v_norm, torch.ones_like(v_norm),
        )

        sinc = (
            torch.sin(safe_v_norm)
            / safe_v_norm
        )

        # taylor approximation near zero:
        # sin(r) / r = 1 - r^2 / 6 + O(r^4)
        sinc = torch.where(
            v_norm > eps,
            sinc, 1.0 - (v_norm ** 2) / 6.0,
        )

        out_blocks = (
            torch.cos(v_norm) * x_blocks + sinc * v_blocks
        )

        out = out_blocks.reshape(
            *out_blocks.shape[:-2], self.ambient_dim,
        )

        # projection removes small floating-point errors.
        return self.project(out)

    def log(self, x, y):
        """
        Product logarithmic map.

        For each circle, compute the shortest signed angular
        difference from x to y and multiply it by the oriented
        unit tangent at x.
        """
        self._check_point(x)
        self._check_point(y)

        # pairwise computations:
        # x: (G, 1, 4)
        # y: (1, P, 4)
        # output: (G, P, 4)
        x, y = torch.broadcast_tensors(x, y)

        x_blocks = self._reshape_blocks(x)
        y_blocks = self._reshape_blocks(y)

        # cos(delta) for every circle factor
        dot = torch.sum(
            x_blocks * y_blocks,
            dim=-1,
        )

        # sin(delta) for every circle factor
        determinant = (
            x_blocks[..., 0] * y_blocks[..., 1]
            - x_blocks[..., 1] * y_blocks[..., 0]
        )

        # Principal signed angle in [-pi, pi].
        delta = torch.atan2(
            determinant,
            dot,
        )

        # If x = (x_1, x_2), then the positively oriented
        # unit tangent is Jx = (-x_2, x_1).
        tangent_basis = torch.stack(
            (
                -x_blocks[..., 1],
                x_blocks[..., 0],
            ),
            dim=-1,
        )

        log_blocks = (
            delta.unsqueeze(-1)
            * tangent_basis
        )

        out = log_blocks.reshape(
            *log_blocks.shape[:-2],
            self.ambient_dim,
        )

        # Numerical safety.
        return self.project_tangent(x, out)

    def dist(self, x, y):
        """
        Product geodesic distance:

            sqrt(delta_1^2 + ... + delta_n^2),

        where each delta_j is the shortest wrapped angular
        difference on one circle.
        """
        self._check_point(x)
        self._check_point(y)

        x, y = torch.broadcast_tensors(x, y)

        x_blocks = self._reshape_blocks(x)
        y_blocks = self._reshape_blocks(y)

        dot = torch.sum(
            x_blocks * y_blocks,
            dim=-1,
        )

        determinant = (
            x_blocks[..., 0] * y_blocks[..., 1]
            - x_blocks[..., 1] * y_blocks[..., 0]
        )

        delta = torch.atan2(
            determinant,
            dot,
        )

        return torch.linalg.norm(
            delta,
            dim=-1,
        )

    # ============================================================
    # Coordinate conversion helpers
    # ============================================================

    def from_angles(self, angles):
        """
        Convert angular coordinates to the ambient embedding.

        Input:
            (..., n_circles)

        Output:
            (..., 2 * n_circles)

        For T^2:

            (theta_1, theta_2)

        becomes

            (cos(theta_1), sin(theta_1),
             cos(theta_2), sin(theta_2)).
        """
        if angles.shape[-1] != self.n_circles:
            raise ValueError(
                f"Expected {self.n_circles} angles, "
                f"but got {angles.shape[-1]}."
            )

        blocks = torch.stack(
            (
                torch.cos(angles),
                torch.sin(angles),
            ),
            dim=-1,
        )

        return blocks.reshape(
            *angles.shape[:-1],
            self.ambient_dim,
        )

    def to_angles(self, x):
        """
        Convert the ambient representation back to angles
        in approximately [-pi, pi].
        """
        self._check_point(x)

        x_blocks = self._reshape_blocks(x)

        return torch.atan2(
            x_blocks[..., 1],
            x_blocks[..., 0],
        )

    def wrap_angles(self, angles):
        """
        Wrap arbitrary angles into the principal interval
        approximately [-pi, pi].
        """
        return torch.atan2(
            torch.sin(angles),
            torch.cos(angles),
        )

    def sample(self, n, device=None, dtype=None):
        """
        Sample uniformly from the flat torus.

        This is a generic manifold sampler, not the future
        checkerboard target sampler.
        """
        angles = (
            2.0
            * torch.pi
            * torch.rand(
                n,
                self.n_circles,
                device=device,
                dtype=dtype,
            )
            - torch.pi
        )

        return self.from_angles(angles)

torus = FlatTorus(n_circles=2)

### Exact \(SO(3)\) geometry

Samples are flattened \(3\times3\) rotation matrices in \(\mathbb R^9\). Exact projection uses the SVD/polar projection, exact distance is the principal rotation angle, and Log/Exp are implemented through the Lie algebra.

In [ ]:
def _hat(w):
    wx, wy, wz = w.unbind(dim=-1)
    z = torch.zeros_like(wx)
    return torch.stack(
        [z, -wz, wy, wz, z, -wx, -wy, wx, z],
        dim=-1,
    ).reshape(*w.shape[:-1], 3, 3)

def _vee(K):
    return torch.stack([K[..., 2, 1], K[..., 0, 2], K[..., 1, 0]], dim=-1)

def _so3_exp_rotvec(w):
    theta2 = (w * w).sum(dim=-1, keepdim=True)
    theta = torch.sqrt(theta2)
    small = theta2 < 1e-8

    A = torch.where(
        small,
        1.0 - theta2 / 6.0 + theta2.square() / 120.0,
        torch.sin(theta) / theta.clamp_min(1e-12),
    )
    B = torch.where(
        small,
        0.5 - theta2 / 24.0 + theta2.square() / 720.0,
        (1.0 - torch.cos(theta)) / theta2.clamp_min(1e-12),
    )

    K = _hat(w)
    I = torch.eye(3, dtype=w.dtype, device=w.device).expand(*w.shape[:-1], 3, 3)
    return I + A.unsqueeze(-1) * K + B.unsqueeze(-1) * (K @ K)

def _matrix_to_quaternion(R):
    m00, m01, m02 = R[..., 0, 0], R[..., 0, 1], R[..., 0, 2]
    m10, m11, m12 = R[..., 1, 0], R[..., 1, 1], R[..., 1, 2]
    m20, m21, m22 = R[..., 2, 0], R[..., 2, 1], R[..., 2, 2]

    qw = 0.5 * torch.sqrt(torch.clamp(1 + m00 + m11 + m22, min=0.0))
    qx = 0.5 * torch.sqrt(torch.clamp(1 + m00 - m11 - m22, min=0.0))
    qy = 0.5 * torch.sqrt(torch.clamp(1 - m00 + m11 - m22, min=0.0))
    qz = 0.5 * torch.sqrt(torch.clamp(1 - m00 - m11 + m22, min=0.0))

    def sgn(x):
        return torch.where(x < 0, -torch.ones_like(x), torch.ones_like(x))

    qx = qx * sgn(m21 - m12)
    qy = qy * sgn(m02 - m20)
    qz = qz * sgn(m10 - m01)

    q = torch.stack([qw, qx, qy, qz], dim=-1)
    q = q / torch.linalg.norm(q, dim=-1, keepdim=True).clamp_min(1e-12)
    q = torch.where(q[..., :1] < 0, -q, q)
    return q

def _so3_log_rotvec(R):
    q = _matrix_to_quaternion(R)
    qw = q[..., 0].clamp(-1.0, 1.0)
    qv = q[..., 1:]
    nv = torch.linalg.norm(qv, dim=-1)
    angle = 2.0 * torch.atan2(nv, qw.clamp_min(0.0))
    scale = torch.where(
        nv > 1e-8,
        angle / nv.clamp_min(1e-12),
        2.0 * torch.ones_like(nv),
    )
    return qv * scale.unsqueeze(-1)

class SO3(Manifold):
    def __init__(self, atol=2e-5, cut_locus_eps=1e-4):
        super().__init__(ambient_dim=9)
        self.manifold_dim = 3
        self.injectivity_radius = math.pi
        self.atol = atol
        self.cut_locus_eps = cut_locus_eps

    def _mat(self, x):
        if x.shape[-1] != 9:
            raise ValueError(f"Expected last dimension 9, got {x.shape[-1]}")
        return x.reshape(*x.shape[:-1], 3, 3)

    def _flat(self, R):
        return R.reshape(*R.shape[:-2], 9)

    def project(self, x):
        A = self._mat(x)
        U, _, Vh = torch.linalg.svd(A)
        det_uv = torch.linalg.det(U @ Vh)
        d = torch.ones(*det_uv.shape, 3, dtype=A.dtype, device=A.device)
        d[..., 2] = torch.where(det_uv < 0, -1.0, 1.0)
        return self._flat(U @ torch.diag_embed(d) @ Vh)

    def project_tangent(self, x, v):
        R, A = self._mat(x), self._mat(v)
        RtA = R.transpose(-1, -2) @ A
        skew = 0.5 * (RtA - RtA.transpose(-1, -2))
        return self._flat(R @ skew)

    def exp(self, x, v):
        R = self._mat(x)
        V = self._mat(self.project_tangent(x, v))
        Omega = R.transpose(-1, -2) @ V
        return self._flat(R @ _so3_exp_rotvec(_vee(Omega)))

    def log(self, x, y):
        x, y = torch.broadcast_tensors(x, y)
        R, S = self._mat(x), self._mat(y)
        Q = R.transpose(-1, -2) @ S
        return self._flat(R @ _hat(_so3_log_rotvec(Q)))

    def dist(self, x, y):
        x, y = torch.broadcast_tensors(x, y)
        R, S = self._mat(x), self._mat(y)
        Q = R.transpose(-1, -2) @ S

        skew_vec = 0.5 * torch.stack(
            [
                Q[..., 2, 1] - Q[..., 1, 2],
                Q[..., 0, 2] - Q[..., 2, 0],
                Q[..., 1, 0] - Q[..., 0, 1],
            ],
            dim=-1,
        )
        sin_theta = torch.linalg.norm(skew_vec, dim=-1)
        cos_theta = (torch.diagonal(Q, dim1=-2, dim2=-1).sum(-1) - 1.0) / 2.0
        return torch.atan2(sin_theta, cos_theta.clamp(-1.0, 1.0))

    def tangent_norm(self, v):
        return torch.linalg.matrix_norm(
            self._mat(v), ord="fro", dim=(-2, -1)
        ) / math.sqrt(2.0)

    def is_on_manifold(self, x):
        R = self._mat(x)
        I = torch.eye(3, dtype=R.dtype, device=R.device)
        orth = torch.linalg.matrix_norm(
            R.transpose(-1, -2) @ R - I, ord="fro", dim=(-2, -1)
        )
        det_err = torch.abs(torch.linalg.det(R) - 1.0)
        return (orth <= self.atol) & (det_err <= self.atol)

    def to_rotvec(self, x):
        return _so3_log_rotvec(self._mat(self.project(x)))



so3 = SO3()

In [ ]:
class Euclidean(Manifold):
    """Ordinary Euclidean geometry in the same ambient representation."""

    def __init__(self, ambient_dim: int):
        if ambient_dim < 1:
            raise ValueError("ambient_dim must be positive.")
        super().__init__(ambient_dim)

    def _check_last_dim(self, x):
        if x.shape[-1] != self.ambient_dim:
            raise ValueError(
                f"Expected last dimension {self.ambient_dim}, got {x.shape[-1]}."
            )

    def project(self, x):
        self._check_last_dim(x)
        return x

    def project_tangent(self, x, v):
        self._check_last_dim(x)
        self._check_last_dim(v)
        x, v = torch.broadcast_tensors(x, v)
        return v

    def exp(self, x, v):
        self._check_last_dim(x)
        self._check_last_dim(v)
        x, v = torch.broadcast_tensors(x, v)
        return x + v

    def log(self, x, y):
        self._check_last_dim(x)
        self._check_last_dim(y)
        x, y = torch.broadcast_tensors(x, y)
        return y - x

    def dist(self, x, y):
        self._check_last_dim(x)
        self._check_last_dim(y)
        x, y = torch.broadcast_tensors(x, y)
        return torch.linalg.norm(y - x, dim=-1)

### Geometry unit tests — run before any expensive training

In [ ]:
def run_geometry_unit_tests():
    torch.manual_seed(123)

    # Sphere.
    xs = sphere.project(torch.randn(128, 3, device=DEVICE))
    ys = sphere.project(torch.randn(128, 3, device=DEVICE))
    vs = sphere.log(xs, ys)
    recs = sphere.exp(xs, vs)
    assert torch.allclose(recs, ys, atol=5e-4, rtol=5e-4)

    # Torus.
    xt = torus.project(torch.randn(128, torus.ambient_dim, device=DEVICE))
    yt = torus.project(torch.randn(128, torus.ambient_dim, device=DEVICE))
    vt = torus.log(xt, yt)
    rect = torus.exp(xt, vt)
    assert torch.allclose(rect, yt, atol=5e-4, rtol=5e-4)

    # SO(3).
    R = so3.project(torch.randn(128, 9, device=DEVICE))
    assert so3.is_on_manifold(R).all()
    w1 = torch.randn(128, 3, device=DEVICE)
    w1 = 1.1 * w1 / torch.linalg.norm(w1, dim=-1, keepdim=True).clamp_min(1e-12)
    w2 = torch.randn(128, 3, device=DEVICE)
    w2 = 0.7 * w2 / torch.linalg.norm(w2, dim=-1, keepdim=True).clamp_min(1e-12)
    R1 = _so3_exp_rotvec(w1).reshape(-1, 9)
    R2 = _so3_exp_rotvec(w2).reshape(-1, 9)
    V = so3.log(R1, R2)
    rec = so3.exp(R1, V)
    assert torch.allclose(rec, R2, atol=6e-4, rtol=6e-4)
    assert torch.allclose(so3.dist(R1, R2), so3.tangent_norm(V), atol=8e-5, rtol=8e-5)

    print('All Sphere / Torus / SO(3) geometry tests passed.')
    print('SO(3) max Exp(Log) absolute error:', float(torch.max(torch.abs(rec - R2))))

run_geometry_unit_tests()

All Sphere / Torus / SO(3) geometry tests passed.
SO(3) max Exp(Log) absolute error: 1.52587890625e-05


## 4. Synthetic targets

In [ ]:
def sample_spiral(n, noise=0.03, turns=4, generator=None):
    """
    Sample a spiral-shaped distribution on the unit sphere.

    Parameters
    ----------
    n : int
        Number of samples.
    noise : float
        Standard deviation of Gaussian noise added before reprojection.
    turns : int
        Number of revolutions around the sphere.
    seed : int or None
        Random seed.

    Returns
    -------
    Tensor of shape (n, 3)
        Points on the unit sphere.
    """
    # uniform parameter along the spiral
    u = torch.rand(n, generator=generator)

    # latitude: north pole -> south pole
    phi = 0.5 * math.pi - math.pi * u

    # longitude: wrap around the sphere
    theta = 2 * math.pi * turns * u

    x = torch.cos(phi) * torch.cos(theta)
    y = torch.cos(phi) * torch.sin(theta)
    z = torch.sin(phi)

    pts = torch.stack([x, y, z], dim=1)

    # add small ambient noise and project back to sphere
    if noise > 0:
        pts = pts + noise * torch.randn(pts.shape, generator=generator)
        pts = pts / torch.linalg.norm(pts, dim=1, keepdim=True)

    return pts

In [ ]:
def sample_checkerboard_torus(n, generator=None, n_tiles=4):
    """
    Sample n points uniformly from the black tiles of a periodic
    checkerboard distribution on the flat torus T^2.

    The torus is represented using two angles:

        (theta_1, theta_2) in [-pi, pi)^2

    and embedded in R^4 as:

        (cos(theta_1), sin(theta_1),
         cos(theta_2), sin(theta_2)).

    Parameters
    ----------
    n : int
        Number of samples.

    seed : int or None
        Optional random seed.

    n_tiles : int
        Number of checkerboard tiles along each axis.
        This should be even so that the checkerboard pattern
        remains consistent across the periodic boundaries.

    Returns
    -------
    samples : torch.Tensor
        Tensor of shape (n, 4) containing points on T^2.
    """

    if n <= 0:
        raise ValueError("n must be positive.")

    if n_tiles <= 0:
        raise ValueError("n_tiles must be positive.")

    if n_tiles % 2 != 0:
        raise ValueError(
            "n_tiles must be even so the checkerboard matches "
            "across the periodic boundaries."
        )

    # --------------------------------------------------------
    # Identify all black tiles
    # --------------------------------------------------------
    #
    # A tile is indexed by (i, j), where:
    #
    #   i = horizontal/theta_1 tile
    #   j = vertical/theta_2 tile
    #
    # We call the tile black when (i + j) is even.

    tile_indices = torch.arange(n_tiles)

    i_grid, j_grid = torch.meshgrid(
        tile_indices,
        tile_indices,
        indexing="ij",
    )

    black_mask = (i_grid + j_grid) % 2 == 0

    black_tiles = torch.stack(
        (
            i_grid[black_mask],
            j_grid[black_mask],
        ),
        dim=-1,
    )

    # For a 4 x 4 checkerboard, black_tiles has shape (8, 2).

    # --------------------------------------------------------
    # Choose a black tile independently for each sample
    # --------------------------------------------------------

    chosen_tile_indices = torch.randint(
        low=0,
        high=black_tiles.shape[0],
        size=(n,),
        generator=generator,
    )

    chosen_tiles = black_tiles[chosen_tile_indices]

    # --------------------------------------------------------
    # Sample uniformly within each selected tile
    # --------------------------------------------------------

    within_tile = torch.rand(
        n,
        2,
        generator=generator,
    )

    tile_width = 2.0 * torch.pi / n_tiles

    angles = (
        -torch.pi
        + (chosen_tiles.to(torch.float32) + within_tile)
        * tile_width
    )

    # angles has shape (n, 2):
    #
    #   angles[:, 0] = theta_1
    #   angles[:, 1] = theta_2

    # Convert the two angles to the R^4 torus embedding.
    samples = torus.from_angles(angles)

    return samples.to(DEVICE)

In [ ]:
SYNTHETIC_CENTRE_ROTVECS = torch.tensor(
    [[1.60, 0.00, 0.00], [0.00, 1.60, 0.00], [0.00, 0.00, 1.60]],
    dtype=torch.float32,
    device=DEVICE,
)
SYNTHETIC_CENTRES = _so3_exp_rotvec(SYNTHETIC_CENTRE_ROTVECS).reshape(-1, 9)
SYNTHETIC_STD = 0.18

def sample_so3_mixture(n, generator=None):
    if generator is None:
        generator = torch.Generator(device="cpu")
        generator.manual_seed(torch.seed())
    mode = torch.randint(0, 3, (n,), generator=generator)
    delta = SYNTHETIC_STD * torch.randn(n, 3, generator=generator)
    centres = SYNTHETIC_CENTRES.index_select(0, mode.to(DEVICE)).reshape(-1, 3, 3)
    local = _so3_exp_rotvec(delta.to(DEVICE))
    return (centres @ local).reshape(-1, 9)

def fixed_so3_synthetic_samples(n, seed):
    g = torch.Generator(device="cpu")
    g.manual_seed(seed)
    return sample_so3_mixture(n, generator=g)

SO3_SYNTHETIC_REFERENCE = fixed_so3_synthetic_samples(4096, 5001)
SO3_SYNTHETIC_VAL = fixed_so3_synthetic_samples(2048, 5002)
SO3_SYNTHETIC_TEST = fixed_so3_synthetic_samples(4096, 5003)

## 5. Dataset specifications — 15 final targets

The target set is deliberately balanced across manifold families:

- **Sphere:** spiral + Volcano + Earthquake + Fire + Flood.
- **Torus:** checkerboard + General + Glycine + Proline + Pre-Pro.
- **SO(3):** synthetic 3-mode mixture + BMLhandball + DanceDB + BMLmovi + CMU.

AMASS uses the prepared **absolute root-orientation** arrays. The preparation script already downsampled to 10 Hz and assigned complete motion sequences to a fixed 80/10/10 train/validation/test split, so neighbouring frames from one motion sequence cannot cross splits.

In [ ]:
@dataclass
class ExperimentSpec:
    name: str
    dataset_family: str
    manifold_kind: str
    manifold: Any
    train_sampler: Callable
    train_data: torch.Tensor
    validation_data: torch.Tensor
    test_data: torch.Tensor
    base_batch_size: int
    target_title: str
    checkerboard_tiles: Optional[int] = None
    source_metadata: Dict[str, Any] = field(default_factory=dict)

    @property
    def intrinsic_dim(self):
        return int(self.manifold.manifold_dim)

    @property
    def k_pca(self):
        return int(3 * self.intrinsic_dim)

    @property
    def effective_batch_size(self):
        return int(min(self.base_batch_size, TRAINING_CONFIG.common_batch_cap))


def tensor_fingerprint(x, seed=9182):
    """Cheap deterministic content fingerprint used only to reject stale caches."""
    x = x.detach().cpu()
    g = torch.Generator(device='cpu').manual_seed(seed)
    n = min(128, len(x))
    idx = torch.randperm(len(x), generator=g)[:n]
    sample = x.index_select(0, idx).contiguous().numpy()
    payload = sample.tobytes() + str(tuple(x.shape)).encode()
    return hashlib.sha256(payload).hexdigest()[:16]


def spec_metadata(spec):
    return {
        'name': spec.name,
        'dataset_family': spec.dataset_family,
        'manifold_kind': spec.manifold_kind,
        'manifold_class': type(spec.manifold).__name__,
        'ambient_dim': int(spec.manifold.ambient_dim),
        'intrinsic_dim': int(spec.intrinsic_dim),
        'k_pca': int(spec.k_pca),
        'n_train': int(len(spec.train_data)),
        'n_validation': int(len(spec.validation_data)),
        'n_test': int(len(spec.test_data)),
        'base_batch_size': int(spec.base_batch_size),
        'effective_batch_size': int(spec.effective_batch_size),
        'target_title': spec.target_title,
        'checkerboard_tiles': spec.checkerboard_tiles,
        'train_fingerprint': tensor_fingerprint(spec.train_data),
        'source_metadata': spec.source_metadata,
    }

In [ ]:
DATASETS = {}

# ---------- Sphere ----------
if RUN_SPHERE:
    spiral_train = sample_target_with_seed(sample_spiral, 10_000, 5101, device=torch.device('cpu'))
    spiral_val = sample_target_with_seed(sample_spiral, 2_048, 5102, device=torch.device('cpu'))
    spiral_test = sample_target_with_seed(sample_spiral, 4_096, 5103, device=torch.device('cpu'))

    DATASETS['sphere_spiral'] = ExperimentSpec(
        name='sphere_spiral', dataset_family='synthetic', manifold_kind='sphere',
        manifold=sphere, train_sampler=sample_spiral,
        train_data=spiral_train.cpu(), validation_data=spiral_val.cpu(), test_data=spiral_test.cpu(),
        base_batch_size=2048, target_title='Spherical spiral',
    )

    EARTH_BATCH_SIZES = {'volcano':128, 'earthquake':980, 'fire':2048, 'flood':780}
    earth_raw = {
        'volcano': load_and_prepare_earth_data(EARTH_DATA_DIR / 'volc.csv', device=torch.device('cpu')),
        'earthquake': load_and_prepare_earth_data(EARTH_DATA_DIR / 'quakes.csv', device=torch.device('cpu')),
        'fire': load_and_prepare_earth_data(EARTH_DATA_DIR / 'fire.csv', device=torch.device('cpu')),
        'flood': load_and_prepare_earth_data(EARTH_DATA_DIR / 'flood.csv', device=torch.device('cpu')),
    }
    for name, data in earth_raw.items():
        train, val, test = split_dataset(data, train_fraction=0.8, val_fraction=0.1, seed=42)
        DATASETS[name] = ExperimentSpec(
            name=name, dataset_family='earth', manifold_kind='sphere', manifold=sphere,
            train_sampler=make_dataset_sampler(train, device=DEVICE), train_data=train.cpu(),
            validation_data=val.cpu(), test_data=test.cpu(), base_batch_size=EARTH_BATCH_SIZES[name],
            target_title=f'{name.capitalize()} distribution',
        )

# ---------- Torus ----------
if RUN_TORUS:
    checker_train = sample_target_with_seed(sample_checkerboard_torus, 10_000, 5201, device=torch.device('cpu'))
    checker_val = sample_target_with_seed(sample_checkerboard_torus, 2_048, 5202, device=torch.device('cpu'))
    checker_test = sample_target_with_seed(sample_checkerboard_torus, 4_096, 5203, device=torch.device('cpu'))

    DATASETS['torus_checkerboard'] = ExperimentSpec(
        name='torus_checkerboard', dataset_family='synthetic', manifold_kind='torus', manifold=torus,
        train_sampler=sample_checkerboard_torus, train_data=checker_train.cpu(),
        validation_data=checker_val.cpu(), test_data=checker_test.cpu(), base_batch_size=2048,
        target_title='Periodic torus checkerboard', checkerboard_tiles=4,
    )

    protein_df = pd.read_csv(require_file(PROTEIN_MASTER_PATH), sep='\t')
    PROTEIN_BATCH_SIZES = {'general':2048, 'glycine':2048, 'proline':1161, 'pre_pro':1056}
    PROTEIN_LABELS = {'general':'General', 'glycine':'Glycine', 'proline':'Proline', 'pre_pro':'Pre-Pro'}

    for name, label in PROTEIN_LABELS.items():
        train_df = protein_df[(protein_df['amino_type']==label) & (protein_df['split']=='train')].copy()
        val_df = protein_df[(protein_df['amino_type']==label) & (protein_df['split']=='validation')].copy()
        test_df = protein_df[(protein_df['amino_type']==label) & (protein_df['split']=='test')].copy()
        train = dataframe_to_torus(train_df, torus, device=torch.device('cpu')).cpu()
        val = dataframe_to_torus(val_df, torus, device=torch.device('cpu')).cpu()
        test = dataframe_to_torus(test_df, torus, device=torch.device('cpu')).cpu()
        DATASETS[name] = ExperimentSpec(
            name=name, dataset_family='protein', manifold_kind='torus', manifold=torus,
            train_sampler=make_dataset_sampler(train, device=DEVICE), train_data=train,
            validation_data=val, test_data=test, base_batch_size=PROTEIN_BATCH_SIZES[name],
            target_title=f'{label} Ramachandran distribution',
        )

# ---------- SO(3) ----------
if RUN_SO3:
    so3_train = fixed_so3_synthetic_samples(10_000, 5301).cpu()
    so3_val = fixed_so3_synthetic_samples(2_048, 5302).cpu()
    so3_test = fixed_so3_synthetic_samples(4_096, 5303).cpu()
    DATASETS['so3_synthetic'] = ExperimentSpec(
        name='so3_synthetic', dataset_family='synthetic', manifold_kind='so3', manifold=so3,
        train_sampler=sample_so3_mixture, train_data=so3_train,
        validation_data=so3_val, test_data=so3_test, base_batch_size=512,
        target_title='Three-mode SO(3) mixture',
    )

    for name, path in SO3_DATASET_FILES.items():
        ds = load_amass_ready(path, SO3_REPRESENTATION)
        # Numerical SO(3) check on deterministic subsets.
        g = torch.Generator(device='cpu').manual_seed(777)
        for split_name in ('train','val','test'):
            arr = ds[split_name]
            n = min(1024, len(arr))
            idx = torch.randperm(len(arr), generator=g)[:n]
            check = arr.index_select(0, idx).to(DEVICE)
            frac = float(so3.is_on_manifold(check).float().mean())
            if frac < 0.999:
                raise RuntimeError(f'{name}/{split_name}: only {frac:.4f} samples satisfy SO(3) checks')

        DATASETS[name] = ExperimentSpec(
            name=name, dataset_family='amass', manifold_kind='so3', manifold=so3,
            train_sampler=make_dataset_sampler(ds['train'], device=DEVICE), train_data=ds['train'],
            validation_data=ds['val'], test_data=ds['test'], base_batch_size=512,
            target_title=f'{name} absolute root orientation',
            source_metadata={
                'ready_file': ds['path'],
                'representation': SO3_REPRESENTATION,
                'split_note': ds['split_note'],
                'representation_note': ds['representation_note'],
            },
        )

print('Available final datasets:')
for name, spec in DATASETS.items():
    print(
        f'  {name:20s} {spec.manifold_kind:6s} '
        f'train={len(spec.train_data):7d} val={len(spec.validation_data):6d} '
        f'test={len(spec.test_data):6d} d={spec.intrinsic_dim} '
        f'kPCA={spec.k_pca} batch={spec.effective_batch_size}'
    )

Available final datasets:
  sphere_spiral        sphere train=  10000 val=  2048 test=  4096 d=2 kPCA=6 batch=256
  volcano              sphere train=    661 val=    82 test=    84 d=2 kPCA=6 batch=128
  earthquake           sphere train=   4896 val=   612 test=   612 d=2 kPCA=6 batch=256
  fire                 sphere train=  10247 val=  1280 test=  1282 d=2 kPCA=6 batch=256
  flood                sphere train=   3900 val=   487 test=   488 d=2 kPCA=6 batch=256
  torus_checkerboard   torus  train=  10000 val=  2048 test=  4096 d=2 kPCA=6 batch=256
  general              torus  train= 108470 val= 18159 test= 11579 d=2 kPCA=6 batch=256
  glycine              torus  train=  10542 val=  1652 test=  1089 d=2 kPCA=6 batch=256
  proline              torus  train=   5806 val=  1179 test=   649 d=2 kPCA=6 batch=256
  pre_pro              torus  train=   5280 val=  1053 test=   577 d=2 kPCA=6 batch=256
  so3_synthetic        so3    train=  10000 val=  2048 test=  4096 d=3 kPCA=9 batch=256
  BMLh

## 6. Geometry-reference construction for Partial and Full

The generator still trains on the original empirical distribution, including repeated observations. Geometry estimation uses a separate training-only support cloud:

\[
X_{\rm train}\to X_{\rm unique}\to X_{\rm geometry}.
\]

If there are more than 2,500 unique training locations, a deterministic candidate subset is coverage-balanced by farthest-point sampling. The local PCA dimension is supplied structurally from the known benchmark family, but the exact manifold formulas do not enter PCA or graph construction.

\[
k_{\rm PCA}=3d = 6\text{ on }S^2,T^2;\qquad k_{\rm PCA}=9\text{ on }SO(3).
\]

In [ ]:
def deduplicate_points(data, tol=None):
    tol = EXPERIMENT_CONFIG.dedup_tol if tol is None else tol
    x = data.detach().cpu().numpy()
    keys = np.rint(x / tol).astype(np.int64)
    _, first_idx = np.unique(keys, axis=0, return_index=True)
    first_idx = np.sort(first_idx)
    return data.index_select(0, torch.tensor(first_idx, dtype=torch.long))


def deterministic_random_subset(data, max_points, seed):
    if len(data) <= max_points:
        return data.clone()
    g = torch.Generator(device='cpu').manual_seed(seed)
    idx = torch.randperm(len(data), generator=g)[:max_points]
    return data.index_select(0, idx)


@torch.no_grad()
def farthest_point_subset(points, n_select):
    if len(points) <= n_select:
        return points.clone()
    x = points.to(DEVICE)
    n_select = min(int(n_select), len(x))
    centre = x.mean(dim=0, keepdim=True)
    first = torch.argmax(((x-centre)**2).sum(dim=-1))
    selected = torch.empty(n_select, dtype=torch.long, device=x.device)
    selected[0] = first
    min_d2 = ((x-x[first])**2).sum(dim=-1)
    min_d2[first] = -1.0
    for t in range(1, n_select):
        nxt = torch.argmax(min_d2)
        selected[t] = nxt
        d2 = ((x-x[nxt])**2).sum(dim=-1)
        min_d2 = torch.minimum(min_d2, d2)
        min_d2[selected[:t+1]] = -1.0
    return x[selected].cpu()


def make_geometry_reference(spec, seed):
    unique = deduplicate_points(spec.train_data)
    meta = {
        'n_train': len(spec.train_data),
        'n_unique': len(unique),
        'duplicate_fraction': 1.0 - len(unique)/max(len(spec.train_data),1),
        'used_fps': False,
    }
    if len(unique) <= EXPERIMENT_CONFIG.geometry_max_points:
        return unique.clone(), meta

    candidate = deterministic_random_subset(
        unique,
        min(EXPERIMENT_CONFIG.fps_candidate_max, len(unique)),
        seed,
    )
    reference = farthest_point_subset(candidate, EXPERIMENT_CONFIG.geometry_max_points)
    meta['used_fps'] = True
    return reference, meta


def _fit_projector(points, intrinsic_dim):
    mean = points.mean(dim=-2)
    centred = points - mean.unsqueeze(-2)
    cov = centred.transpose(-1,-2) @ centred / max(points.shape[-2]-1, 1)
    eigvals, eigvecs = torch.linalg.eigh(cov)
    U = eigvecs[..., :, -intrinsic_dim:]
    P = U @ U.transpose(-1,-2)
    return mean, P, eigvals


def build_atlas(reference, intrinsic_dim, k_pca):
    reference = reference.to(DEVICE)
    if len(reference) <= k_pca:
        raise ValueError(f'Need more than k_pca={k_pca} unique geometry points; got {len(reference)}')
    d = torch.cdist(reference, reference)
    d.fill_diagonal_(torch.inf)
    _, idx = torch.topk(d, k=k_pca, dim=1, largest=False, sorted=True)
    neighbours = reference[idx]
    mean, P, eigvals = _fit_projector(neighbours, intrinsic_dim)
    return {'centres':reference, 'means':mean, 'projectors':P, 'eigvals':eigvals}


def learned_projection_once(query, atlas, num_charts=None):
    num_charts = EXPERIMENT_CONFIG.projection_num_charts if num_charts is None else num_charts
    centres = atlas['centres']
    with torch.no_grad():
        route = torch.cdist(query.detach(), centres)
        m = min(num_charts, len(centres))
        idx = torch.topk(route, k=m, dim=1, largest=False, sorted=True).indices
    c = centres[idx]
    mu = atlas['means'][idx]
    P = atlas['projectors'][idx]
    live = torch.linalg.norm(query.unsqueeze(1)-c, dim=-1)
    scale = live[:,-1:].detach().clamp_min(1e-4)
    weights = torch.softmax(-(live/scale).pow(2), dim=1)
    offset = query.unsqueeze(1)-mu
    projected_offset = torch.einsum('bmij,bmj->bmi', P, offset)
    local = mu + projected_offset
    return (weights.unsqueeze(-1)*local).sum(dim=1)


def learned_projection(query, atlas):
    out = query
    for _ in range(EXPERIMENT_CONFIG.projection_iterations):
        out = learned_projection_once(out, atlas)
    return out

## 7. Graph construction and out-of-sample graph distance

In [ ]:
def ambient_pairwise_numpy(points):
    # 2,500^2 float32 is ~25 MB. Compute once per dataset geometry base.
    return torch.cdist(points.to(DEVICE), points.to(DEVICE)).cpu().numpy().astype(np.float32)


def knn_order_from_pairwise(pairwise, max_k):
    work = pairwise.copy()
    np.fill_diagonal(work, np.inf)
    max_k = min(int(max_k), work.shape[0]-1)
    idx = np.argpartition(work, kth=max_k-1, axis=1)[:,:max_k]
    d = np.take_along_axis(work, idx, axis=1)
    order = np.argsort(d, axis=1)
    return np.take_along_axis(idx, order, axis=1)


def directed_knn_graph(pairwise, order, k):
    n = pairwise.shape[0]
    cols = order[:,:k].reshape(-1)
    rows = np.repeat(np.arange(n), k)
    vals = np.maximum(pairwise[rows,cols].astype(np.float64), 1e-12)
    return csr_matrix((vals,(rows,cols)), shape=(n,n))


def mutual_knn_graph(directed):
    presence = directed.T.copy()
    presence.data = np.ones_like(presence.data)
    graph = directed.multiply(presence)
    graph = graph.maximum(graph.T)
    graph.eliminate_zeros()
    return graph


@dataclass
class GeometryBase:
    reference: torch.Tensor
    atlas: Dict[str, torch.Tensor]
    pairwise: np.ndarray
    order: np.ndarray
    metadata: Dict[str, Any]


@dataclass
class GraphContext:
    dataset: str
    k_graph: int
    reference: torch.Tensor
    atlas: Dict[str, torch.Tensor]
    graph_distance: torch.Tensor
    graph_radius: torch.Tensor
    graph_components: int
    graph_lcc_fraction: float
    graph_finite_fraction: float
    median_graph_distance: float


def geometry_dir(spec):
    return RESULTS_ROOT / spec.manifold_kind / spec.name / 'geometry'


def _geometry_base_signature(spec):
    return config_signature({
        'format': EXPERIMENT_FORMAT_VERSION,
        'phase': 'geometry_base',
        'spec': spec_metadata(spec),
        'geometry_max_points': EXPERIMENT_CONFIG.geometry_max_points,
        'dedup_tol': EXPERIMENT_CONFIG.dedup_tol,
        'fps_candidate_max': EXPERIMENT_CONFIG.fps_candidate_max,
        'k_pca': spec.k_pca,
        'projection_num_charts': EXPERIMENT_CONFIG.projection_num_charts,
        'projection_iterations': EXPERIMENT_CONFIG.projection_iterations,
        'max_k_graph': max(EXPERIMENT_CONFIG.k_graph_grid),
    })


def build_or_load_geometry_base(spec, resume=RESUME, force=FORCE_RERUN):
    out = geometry_dir(spec)
    out.mkdir(parents=True, exist_ok=True)
    path = out / 'base.pt'
    signature = _geometry_base_signature(spec)

    if resume and not force and path.exists():
        saved = safe_torch_load(path)
        if saved.get('run_signature') == signature:
            reference = saved['reference'].to(DEVICE)
            atlas = {k:(v.to(DEVICE) if torch.is_tensor(v) else v) for k,v in saved['atlas'].items()}
            return GeometryBase(reference, atlas, saved['pairwise'], saved['order'], saved['metadata'])

    reference_cpu, ref_meta = make_geometry_reference(spec, seed=7000 + sorted(DATASETS).index(spec.name))
    reference = reference_cpu.to(DEVICE)
    atlas = build_atlas(reference, spec.intrinsic_dim, spec.k_pca)
    pairwise = ambient_pairwise_numpy(reference)
    valid_grid = [k for k in EXPERIMENT_CONFIG.k_graph_grid if k < len(reference)]
    if not valid_grid:
        raise RuntimeError(f'{spec.name}: no valid k_graph candidates for reference size {len(reference)}')
    order = knn_order_from_pairwise(pairwise, max(max(valid_grid), EXPERIMENT_CONFIG.graph_num_anchors))
    metadata = {
        **ref_meta,
        'geometry_reference_n': len(reference),
        'intrinsic_dim': spec.intrinsic_dim,
        'k_pca': spec.k_pca,
        'valid_k_graph_grid': valid_grid,
    }

    payload = {
        'run_signature': signature,
        'reference': reference.cpu(),
        'atlas': {k:(v.cpu() if torch.is_tensor(v) else v) for k,v in atlas.items()},
        'pairwise': pairwise,
        'order': order,
        'metadata': metadata,
    }
    atomic_torch_save(payload, path)
    atomic_json_save(metadata, out / 'base_summary.json')
    return GeometryBase(reference, atlas, pairwise, order, metadata)


def _graph_signature(spec, base, k_graph):
    return config_signature({
        'format': EXPERIMENT_FORMAT_VERSION,
        'phase': 'graph_context',
        'spec': spec_metadata(spec),
        'geometry_metadata': base.metadata,
        'k_graph': int(k_graph),
        'graph_mode': EXPERIMENT_CONFIG.graph_mode,
    })


def build_or_load_graph_context(spec, k_graph, base=None, resume=RESUME, force=FORCE_RERUN):
    if EXPERIMENT_CONFIG.graph_mode != 'mutual':
        raise ValueError('Locked protocol requires mutual kNN graph.')
    base = build_or_load_geometry_base(spec, resume=resume, force=force) if base is None else base
    if k_graph >= len(base.reference):
        raise ValueError(f'k_graph={k_graph} invalid for reference size {len(base.reference)}')

    out = geometry_dir(spec) / f'graph_k_{k_graph}'
    out.mkdir(parents=True, exist_ok=True)
    path = out / 'context.pt'
    signature = _graph_signature(spec, base, k_graph)

    if resume and not force and path.exists():
        saved = safe_torch_load(path)
        if saved.get('run_signature') == signature:
            return GraphContext(
                dataset=spec.name, k_graph=k_graph,
                reference=base.reference, atlas=base.atlas,
                graph_distance=saved['graph_distance'].to(DEVICE),
                graph_radius=saved['graph_radius'].to(DEVICE),
                graph_components=int(saved['graph_components']),
                graph_lcc_fraction=float(saved['graph_lcc_fraction']),
                graph_finite_fraction=float(saved['graph_finite_fraction']),
                median_graph_distance=float(saved['median_graph_distance']),
            )

    directed = directed_knn_graph(base.pairwise, base.order, k_graph)
    graph = mutual_knn_graph(directed)
    ncomp, labels = connected_components(graph, directed=False, return_labels=True)
    counts = np.bincount(labels)
    lcc = float(counts.max()/len(labels))
    gd_np = shortest_path(graph, directed=False, method='D').astype(np.float32)
    finite_mask = np.isfinite(gd_np)
    finite_fraction = float(finite_mask.mean())
    finite_vals = gd_np[finite_mask & (gd_np > 1e-8)]
    if finite_vals.size == 0:
        raise RuntimeError(f'{spec.name}, k={k_graph}: graph has no finite non-zero paths')
    median_graph = float(np.median(finite_vals))

    # Pairwise matrix includes self distance at column 0; sorted column k is kth non-self neighbour.
    kth = np.sort(base.pairwise, axis=1)[:, k_graph]
    radius = torch.tensor(kth, dtype=torch.float32)

    saved = {
        'run_signature': signature,
        'graph_distance': torch.from_numpy(gd_np),
        'graph_radius': radius,
        'graph_components': int(ncomp),
        'graph_lcc_fraction': lcc,
        'graph_finite_fraction': finite_fraction,
        'median_graph_distance': median_graph,
    }
    atomic_torch_save(saved, path)
    atomic_json_save({k:v for k,v in saved.items() if k not in {'graph_distance','graph_radius'}}, out / 'summary.json')

    return GraphContext(
        dataset=spec.name, k_graph=k_graph, reference=base.reference, atlas=base.atlas,
        graph_distance=torch.from_numpy(gd_np).to(DEVICE), graph_radius=radius.to(DEVICE),
        graph_components=int(ncomp), graph_lcc_fraction=lcc,
        graph_finite_fraction=finite_fraction, median_graph_distance=median_graph,
    )


def query_graph_anchors(query, ctx):
    d = torch.cdist(query, ctx.reference)
    requested = min(max(EXPERIMENT_CONFIG.graph_num_anchors, ctx.k_graph), len(ctx.reference))
    values, idx = torch.topk(d, k=requested, dim=1, largest=False, sorted=True)
    return {
        'idx': idx[:,:EXPERIMENT_CONFIG.graph_num_anchors],
        'dist': values[:,:EXPERIMENT_CONFIG.graph_num_anchors],
        'radius': values[:,ctx.k_graph-1],
    }


def pairwise_graph_distance(x, y, ctx, anchors_x=None, anchors_y=None):
    # Memory-safe min over the small anchor cross-product.
    ax = query_graph_anchors(x, ctx) if anchors_x is None else anchors_x
    ay = query_graph_anchors(y, ctx) if anchors_y is None else anchors_y
    B, M = len(x), len(y)
    route = torch.full((B,M), torch.inf, device=x.device, dtype=x.dtype)
    for a in range(ax['idx'].shape[1]):
        ia, da = ax['idx'][:,a], ax['dist'][:,a]
        for b in range(ay['idx'].shape[1]):
            ib, db = ay['idx'][:,b], ay['dist'][:,b]
            middle = ctx.graph_distance[ia[:,None], ib[None,:]]
            route = torch.minimum(route, da[:,None] + middle + db[None,:])

    # Direct local chord is allowed when the two query points are inside the
    # local graph scale; otherwise the graph path is used.
    chord = torch.cdist(x,y)
    local_threshold = torch.maximum(ax['radius'][:,None], ay['radius'][None,:])
    direct = torch.where(chord <= local_threshold, chord, torch.full_like(chord, torch.inf))
    return torch.minimum(route, direct)


def nearest_atlas_projector(query, ctx):
    with torch.no_grad():
        idx = torch.cdist(query.detach(), ctx.atlas['centres']).argmin(dim=1)
    return ctx.atlas['projectors'][idx]


def pca_direction_with_magnitude(x, y, magnitude, ctx):
    P = nearest_atlas_projector(x, ctx)
    ambient = y.unsqueeze(0) - x.unsqueeze(1)
    projected = torch.einsum('bij,bmj->bmi', P, ambient)
    norm = torch.linalg.norm(projected, dim=-1, keepdim=True)
    direction = projected / norm.clamp_min(1e-10)
    out = magnitude.unsqueeze(-1) * direction
    invalid = (norm <= 1e-10) | (~torch.isfinite(magnitude).unsqueeze(-1))
    return torch.where(invalid, torch.zeros_like(out), out)

## 8. Drift fields and the four model definitions

All variants use the same batch-normalised Laplace-kernel construction. The positive/negative displacement field is formed first, then a common step safeguard is applied, and finally each method constructs its own stop-gradient target.

In [ ]:
def _masked_softmax(logits, valid_mask, dim):
    masked = logits.masked_fill(~valid_mask, -torch.inf)
    p = torch.softmax(masked, dim=dim)
    p = torch.where(valid_mask, p, torch.zeros_like(p))
    return torch.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)


def compute_drift_exact(gen, pos, temp, manifold):
    if temp <= 0:
        raise ValueError('temperature must be positive')
    gen = manifold.project(gen)
    pos = manifold.project(pos)
    B, M = len(gen), len(pos)
    d_gp = manifold.dist(gen.unsqueeze(1), pos.unsqueeze(0))
    d_gg = manifold.dist(gen.unsqueeze(1), gen.unsqueeze(0))
    valid_gp = torch.ones_like(d_gp, dtype=torch.bool)
    valid_gg = ~torch.eye(B, dtype=torch.bool, device=gen.device)

    # Principal log is undefined at the cut locus. Such exact pairs have measure
    # zero, but masking protects synthetic/adversarial coincidences.
    if isinstance(manifold, Sphere):
        valid_gp &= d_gp < manifold.injectivity_radius - manifold.atol
        valid_gg &= d_gg < manifold.injectivity_radius - manifold.atol
    if isinstance(manifold, SO3):
        valid_gp &= d_gp < manifold.injectivity_radius - manifold.cut_locus_eps
        valid_gg &= d_gg < manifold.injectivity_radius - manifold.cut_locus_eps

    logits = torch.cat([-d_gp/temp, -d_gg/temp], dim=1)
    valid = torch.cat([valid_gp, valid_gg], dim=1)
    row = _masked_softmax(logits, valid, dim=-1)
    col = _masked_softmax(logits, valid, dim=-2)
    K = torch.sqrt(row*col)
    Kgp, Kgg = torch.split(K, [M,B], dim=1)
    delta_gp = manifold.log(gen.unsqueeze(1), pos.unsqueeze(0))
    delta_gg = manifold.log(gen.unsqueeze(1), gen.unsqueeze(0))
    pos_mass = Kgg.sum(-1, keepdim=True)
    neg_mass = Kgp.sum(-1, keepdim=True)
    Vp = ((Kgp*pos_mass).unsqueeze(-1)*delta_gp).sum(1)
    Vq = ((Kgg*neg_mass).unsqueeze(-1)*delta_gg).sum(1)
    return manifold.project_tangent(gen, Vp-Vq), 1.0


def compute_drift_graph(gen, pos, temp, ctx):
    B, M = len(gen), len(pos)
    ag = query_graph_anchors(gen, ctx)
    ap = query_graph_anchors(pos, ctx)
    d_gp = pairwise_graph_distance(gen, pos, ctx, ag, ap)
    d_gg = pairwise_graph_distance(gen, gen, ctx, ag, ag)
    valid_gp = torch.isfinite(d_gp)
    valid_gg = torch.isfinite(d_gg) & ~torch.eye(B, dtype=torch.bool, device=gen.device)
    logits = torch.cat([-d_gp/temp, -d_gg/temp], dim=1)
    valid = torch.cat([valid_gp, valid_gg], dim=1)
    row = _masked_softmax(logits, valid, dim=-1)
    col = _masked_softmax(logits, valid, dim=-2)
    K = torch.sqrt(row*col)
    Kgp, Kgg = torch.split(K, [M,B], dim=1)
    delta_gp = pca_direction_with_magnitude(gen, pos, d_gp, ctx)
    delta_gg = pca_direction_with_magnitude(gen, gen, d_gg, ctx)
    pos_mass = Kgg.sum(-1, keepdim=True)
    neg_mass = Kgp.sum(-1, keepdim=True)
    Vp = ((Kgp*pos_mass).unsqueeze(-1)*delta_gp).sum(1)
    Vq = ((Kgg*neg_mass).unsqueeze(-1)*delta_gg).sum(1)
    finite_fraction = float(torch.cat([valid_gp.flatten(), valid_gg.flatten()]).float().mean())
    return Vp-Vq, finite_fraction


def compute_drift_euclidean(gen, pos, temp):
    B, M = len(gen), len(pos)
    d_gp = torch.cdist(gen,pos)
    d_gg = torch.cdist(gen,gen)
    valid_gp = torch.ones_like(d_gp, dtype=torch.bool)
    valid_gg = ~torch.eye(B, dtype=torch.bool, device=gen.device)
    logits = torch.cat([-d_gp/temp, -d_gg/temp], dim=1)
    valid = torch.cat([valid_gp, valid_gg], dim=1)
    row = _masked_softmax(logits, valid, dim=-1)
    col = _masked_softmax(logits, valid, dim=-2)
    K = torch.sqrt(row*col)
    Kgp, Kgg = torch.split(K,[M,B],dim=1)
    delta_gp = pos.unsqueeze(0)-gen.unsqueeze(1)
    delta_gg = gen.unsqueeze(0)-gen.unsqueeze(1)
    pos_mass = Kgg.sum(-1,keepdim=True)
    neg_mass = Kgp.sum(-1,keepdim=True)
    Vp = ((Kgp*pos_mass).unsqueeze(-1)*delta_gp).sum(1)
    Vq = ((Kgg*neg_mass).unsqueeze(-1)*delta_gg).sum(1)
    return Vp-Vq, 1.0


def initial_map(raw, method, spec, ctx=None):
    if method in ('exact','partial'):
        return spec.manifold.project(raw)
    if method == 'full':
        return learned_projection(raw, ctx.atlas)
    if method == 'euclidean':
        return raw
    raise ValueError(method)


def compute_method_drift(gen, pos, method, spec, temperature, ctx=None):
    if method == 'exact':
        return compute_drift_exact(gen, pos, temperature, spec.manifold)
    if method in ('partial','full'):
        return compute_drift_graph(gen, pos, temperature, ctx)
    if method == 'euclidean':
        return compute_drift_euclidean(gen, pos, temperature)
    raise ValueError(method)


def method_step_norm(V, method, spec):
    # Exact uses the exact tangent norm where it differs from ambient norm
    # (notably SO(3)); graph/Euclidean vectors live in their own ambient/graph units.
    if method == 'exact' and hasattr(spec.manifold, 'tangent_norm'):
        return spec.manifold.tangent_norm(V)
    return torch.linalg.norm(V, dim=-1)


def cap_method_step(V, method, spec, max_step):
    norms = TRAINING_CONFIG.eta * method_step_norm(V, method, spec)
    scale = torch.clamp(max_step / norms.clamp_min(1e-12), max=1.0)
    return V*scale.unsqueeze(-1), float((scale<1).float().mean())


def construct_target(gen, V, method, spec, ctx=None):
    step = TRAINING_CONFIG.eta * V
    if method == 'exact':
        step = spec.manifold.project_tangent(gen, step)
        return spec.manifold.exp(gen, step)
    if method == 'partial':
        return spec.manifold.project(gen + step)
    if method == 'full':
        return learned_projection(gen + step, ctx.atlas)
    if method == 'euclidean':
        return gen + step
    raise ValueError(method)

## 9. Network, EMA, learning-rate schedule, and loss utilities

In [ ]:
class MLPDrift(nn.Module):
    def __init__(self, in_dim, hidden_dim, ambient_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, ambient_dim),
        )
    def forward(self, z):
        return self.net(z)


@torch.no_grad()
def update_ema_model(ema_model, model, decay):
    for pe,p in zip(ema_model.parameters(), model.parameters()):
        pe.mul_(decay).add_(p, alpha=1.0-decay)
    for be,b in zip(ema_model.buffers(), model.buffers()):
        be.copy_(b)


def cosine_learning_rate(step, base_lr, min_lr, decay_start_step, max_steps):
    if step <= decay_start_step:
        return base_lr
    if max_steps <= decay_start_step:
        return min_lr
    progress = min(max((step-decay_start_step)/(max_steps-decay_start_step),0.0),1.0)
    cosine = 0.5*(1.0+math.cos(math.pi*progress))
    return min_lr + (base_lr-min_lr)*cosine


def common_fixed_point_loss(gen, target):
    return (gen-target.detach()).pow(2).sum(dim=-1).mean()

## 10. Common validation and final evaluation metrics

**Model selection is geometry-blind.** Every method is validated on the raw samples it actually produces using Gaussian MMD² in the common ambient representation. Its bandwidth is the median ambient pairwise distance of the *training data only*.

Final evaluation deliberately asks two questions:

1. **Without oracle correction:** raw ambient MMD² and raw support error.
2. **After oracle projection for scientific benchmarking:** projected ambient MMD² and exact intrinsic geodesic-kernel discrepancy².

This makes the Euclidean baseline meaningful both when the manifold is unknown and when one grants it an oracle post-processing projection.

In [ ]:
@torch.no_grad()
def ambient_gaussian_mmd_sq(real, gen, bandwidth):
    drr = torch.cdist(real,real)
    dgg = torch.cdist(gen,gen)
    drg = torch.cdist(real,gen)
    return float((
        torch.exp(-(drr/bandwidth).pow(2)).mean()
        + torch.exp(-(dgg/bandwidth).pow(2)).mean()
        - 2.0*torch.exp(-(drg/bandwidth).pow(2)).mean()
    ).item())


@torch.no_grad()
def _intrinsic_kernel_mean(a, b, manifold, bandwidth=1.0, chunk_size=256):
    total = 0.0
    count = 0
    for start in range(0, len(a), chunk_size):
        aa = a[start:start+chunk_size]
        d = manifold.dist(aa.unsqueeze(1), b.unsqueeze(0))
        total += float(torch.exp(-(d/bandwidth).pow(2)).sum().item())
        count += int(d.numel())
    return total / max(count, 1)


@torch.no_grad()
def intrinsic_kernel_discrepancy_sq(real, projected, manifold, bandwidth=1.0):
    # Chunked row-wise calculation avoids constructing an N x N x ambient_dim
    # broadcast tensor for SO(3) at the full 2,048-sample evaluation size.
    real = manifold.project(real)
    projected = manifold.project(projected)
    krr = _intrinsic_kernel_mean(real, real, manifold, bandwidth)
    kgg = _intrinsic_kernel_mean(projected, projected, manifold, bandwidth)
    krg = _intrinsic_kernel_mean(real, projected, manifold, bandwidth)
    return float(krr + kgg - 2.0*krg)


@torch.no_grad()
def support_distance(samples, manifold):
    if isinstance(manifold, Sphere):
        return (torch.linalg.norm(samples,dim=-1)-1.0).abs()
    if isinstance(manifold, FlatTorus):
        blocks = samples.reshape(*samples.shape[:-1], manifold.n_circles, 2)
        radii = torch.linalg.norm(blocks, dim=-1)
        return torch.linalg.norm(radii-1.0, dim=-1)
    if isinstance(manifold, SO3):
        projected = manifold.project(samples)
        A = samples.reshape(*samples.shape[:-1],3,3)
        P = projected.reshape(*projected.shape[:-1],3,3)
        return torch.linalg.matrix_norm(A-P, ord='fro', dim=(-2,-1))
    raise NotImplementedError(type(manifold).__name__)


@torch.no_grad()
def checkerboard_accuracy(samples, manifold, n_tiles=4):
    angles = manifold.to_angles(manifold.project(samples))
    a = torch.remainder(angles + torch.pi, 2*torch.pi)
    width = 2*torch.pi/n_tiles
    idx = torch.floor(a/width).long()%n_tiles
    return float((((idx[:,0]+idx[:,1])%2)==0).float().mean())


@torch.no_grad()
def so3_constraint_diagnostics(samples):
    A = samples.reshape(-1,3,3)
    I = torch.eye(3,dtype=A.dtype,device=A.device)
    orth = torch.linalg.matrix_norm(A.transpose(-1,-2)@A-I,ord='fro',dim=(-2,-1))
    det_err = torch.abs(torch.linalg.det(A)-1.0)
    return {
        'so3_orthogonality_mean': float(orth.mean()),
        'so3_orthogonality_p95': float(torch.quantile(orth,0.95)),
        'so3_det_abs_error_mean': float(det_err.mean()),
        'so3_det_abs_error_p95': float(torch.quantile(det_err,0.95)),
    }

## 11. Reproducible median scales and fixed evaluation objects

In [ ]:
@torch.no_grad()
def estimate_median_pairwise_distance(data, distance_fn, num_pairs=None, seed=None, chunk_size=4096):
    num_pairs = EXPERIMENT_CONFIG.median_num_pairs if num_pairs is None else num_pairs
    seed = EXPERIMENT_CONFIG.median_pair_seed if seed is None else seed
    if len(data) < 2:
        raise ValueError('At least two observations are required')
    g = torch.Generator(device='cpu').manual_seed(seed)
    i = torch.randint(0,len(data),(num_pairs,),generator=g)
    j = torch.randint(0,len(data),(num_pairs,),generator=g)
    same = i==j
    j[same]=(j[same]+1)%len(data)
    vals=[]
    for start in range(0,num_pairs,chunk_size):
        ia=i[start:start+chunk_size]; ib=j[start:start+chunk_size]
        a=data.index_select(0,ia).to(DEVICE)
        b=data.index_select(0,ib).to(DEVICE)
        d=distance_fn(a,b)
        vals.append(d.detach().cpu())
    d=torch.cat(vals)
    d=d[torch.isfinite(d)&(d>1e-8)]
    if not len(d):
        raise RuntimeError('No finite nonzero distances')
    return float(d.median())


def fixed_subset(dataset, n, seed):
    n=min(int(n),len(dataset))
    g=torch.Generator(device='cpu').manual_seed(seed)
    idx=torch.randperm(len(dataset),generator=g)[:n]
    return dataset.index_select(0,idx).to(DEVICE)


def fixed_noise(n, seed):
    g=torch.Generator(device='cpu').manual_seed(seed)
    return torch.randn(n,TRAINING_CONFIG.latent_dim,generator=g).to(DEVICE)


def prepare_eval_objects(spec):
    nval=min(EXPERIMENT_CONFIG.validation_eval_max,len(spec.validation_data))
    ntest=min(EXPERIMENT_CONFIG.test_eval_max,len(spec.test_data))
    val=fixed_subset(spec.validation_data,nval,EXPERIMENT_CONFIG.validation_subset_seed)
    test=fixed_subset(spec.test_data,ntest,EXPERIMENT_CONFIG.test_subset_seed)
    val_noise=[fixed_noise(nval,s) for s in EXPERIMENT_CONFIG.validation_noise_seeds]
    test_noise=[fixed_noise(ntest,s) for s in EXPERIMENT_CONFIG.test_noise_seeds]
    return val,test,val_noise,test_noise


def compute_dataset_scales(spec):
    exact_median=estimate_median_pairwise_distance(
        spec.train_data,
        lambda a,b: spec.manifold.dist(spec.manifold.project(a),spec.manifold.project(b)),
    )
    ambient_median=estimate_median_pairwise_distance(
        spec.train_data,
        lambda a,b: torch.linalg.norm(a-b,dim=-1),
    )
    return {
        'exact_median_distance': exact_median,
        'ambient_median_distance': ambient_median,
        'ambient_eval_bandwidth': ambient_median,
    }

## 12. Validation, training, and best-checkpoint restoration

In [ ]:
@torch.no_grad()
def generate_raw_method(model, noise, method, spec, ctx=None):
    model.eval()
    return initial_map(model(noise), method, spec, ctx)


@torch.no_grad()
def validation_score(model, method, spec, ctx, val_data, val_noise_batches, ambient_bandwidth):
    scores=[]
    for z in val_noise_batches:
        raw=generate_raw_method(model,z,method,spec,ctx)
        scores.append(ambient_gaussian_mmd_sq(val_data,raw,ambient_bandwidth))
    return float(np.mean(scores)), float(np.std(scores,ddof=1) if len(scores)>1 else 0.0)


def _meaningful_improvement(current, reference, rel_delta, abs_delta):
    if not math.isfinite(reference):
        return True
    threshold=max(abs(reference)*rel_delta,abs_delta)
    return current < reference-threshold


def train_one_run(spec, method, temperature, seed, val_data, val_noise_batches, ambient_bandwidth, ctx=None):
    cfg=TRAINING_CONFIG
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model=MLPDrift(cfg.latent_dim,cfg.hidden_dim,spec.manifold.ambient_dim).to(DEVICE)
    ema=copy.deepcopy(model).to(DEVICE).eval()
    for p in ema.parameters(): p.requires_grad_(False)
    opt=torch.optim.Adam(model.parameters(),lr=cfg.learning_rate)
    ng=torch.Generator(device='cpu').manual_seed(seed+10)
    dg=torch.Generator(device='cpu').manual_seed(seed+20)

    best_score=float('inf'); best_state=None; best_step=None
    patience_ref=float('inf'); bad=0; stopped_early=False; stop_step=cfg.max_steps
    history={
        'step':[],'loss':[],'smoothed_loss':[],'gradient_norm':[],'cap_fraction':[],
        'graph_finite_fraction':[],'support_mean':[],'learning_rate':[],'validation':[],
    }
    smoothed=None
    batch=spec.effective_batch_size

    pbar=tqdm(range(1,cfg.max_steps+1),desc=f'{spec.name} | {METHOD_LABELS[method]} | seed={seed} | tau={temperature:.4g}')
    for step in pbar:
        lr=cosine_learning_rate(step,cfg.learning_rate,cfg.min_learning_rate,cfg.lr_decay_start_step,cfg.max_steps)
        for group in opt.param_groups: group['lr']=lr

        pos=spec.train_sampler(batch,generator=dg).to(DEVICE)
        z=torch.randn(batch,cfg.latent_dim,generator=ng).to(DEVICE)
        raw=model(z)
        gen=initial_map(raw,method,spec,ctx)

        with torch.no_grad():
            V,finite_fraction=compute_method_drift(gen,pos,method,spec,temperature,ctx)
            if not torch.isfinite(V).all():
                raise FloatingPointError(f'Non-finite drift: {spec.name}/{method}/step={step}')
            V,cap_fraction=cap_method_step(V,method,spec,cfg.max_step)
            target=construct_target(gen,V,method,spec,ctx)

        loss=common_fixed_point_loss(gen,target)
        if not torch.isfinite(loss):
            raise FloatingPointError(f'Non-finite loss: {spec.name}/{method}/step={step}')
        opt.zero_grad(set_to_none=True)
        loss.backward()
        grad_norm=float(torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.gradient_clip_norm))
        opt.step()
        update_ema_model(ema,model,cfg.ema_decay)

        loss_value=float(loss.item())
        smoothed=loss_value if smoothed is None else 0.96*smoothed+0.04*loss_value

        if step==1 or step%cfg.history_every==0:
            with torch.no_grad():
                support_mean=float(support_distance(gen,spec.manifold).mean())
            history['step'].append(step); history['loss'].append(loss_value); history['smoothed_loss'].append(smoothed)
            history['gradient_norm'].append(grad_norm); history['cap_fraction'].append(cap_fraction)
            history['graph_finite_fraction'].append(finite_fraction); history['support_mean'].append(support_mean)
            history['learning_rate'].append(lr)

        if step%cfg.validate_every==0 or step==cfg.max_steps:
            score,score_std=validation_score(ema,method,spec,ctx,val_data,val_noise_batches,ambient_bandwidth)
            history['validation'].append({'step':step,'raw_ambient_mmd_sq':score,'std_noise':score_std})
            if score<best_score:
                best_score=score; best_step=step; best_state=copy.deepcopy(ema.state_dict())
            if step>=cfg.early_stopping_start_step:
                if _meaningful_improvement(score,patience_ref,cfg.min_delta_rel,cfg.min_delta_abs):
                    patience_ref=score; bad=0
                else:
                    bad+=1
                if bad>=cfg.early_stopping_patience:
                    stopped_early=True; stop_step=step; break

        pbar.set_postfix(loss=f'{smoothed:.2e}',cap=f'{cap_fraction:.0%}',finite=f'{finite_fraction:.0%}',lr=f'{lr:.1e}')

    if best_state is None:
        raise RuntimeError('No validation checkpoint was recorded')
    ema.load_state_dict(best_state)
    ema.eval()
    return {
        'model':ema,'best_validation_score':float(best_score),'best_step':int(best_step),
        'stop_step':int(stop_step),'stopped_early':bool(stopped_early),'history':history,
    }

## 13. Atomic saving, cache signatures, and run paths

In [ ]:
def _to_builtin(obj):
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, torch.Tensor):
        if obj.numel() == 1:
            return obj.item()
        return obj.detach().cpu().tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, dict):
        return {str(k): _to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_builtin(v) for v in obj]
    if hasattr(obj, "__dataclass_fields__"):
        return _to_builtin(asdict(obj))
    return str(obj)


def config_signature(payload) -> str:
    serialised = json.dumps(
        _to_builtin(payload),
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(serialised.encode("utf-8")).hexdigest()[:20]


def safe_torch_load(path):
    path = Path(path)
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def atomic_torch_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)


def atomic_json_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(_to_builtin(obj), f, indent=2, sort_keys=True)
    tmp.replace(path)


def maybe_load_run(path, expected_signature, resume=True, force=False):
    path = Path(path)
    if force or not resume or not path.exists():
        return None

    loaded = safe_torch_load(path)
    if loaded.get("run_signature") != expected_signature:
        print(f"Ignoring stale saved run with mismatched signature: {path}")
        return None

    print(f"Resuming from: {path}")
    return loaded


def cpu_state_dict(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }

In [ ]:
def experiment_dir(spec):
    return RESULTS_ROOT / spec.manifold_kind / spec.name


def method_dir(spec, method):
    return experiment_dir(spec) / method


def _candidate_slug(k_graph, c):
    cslug=f'{float(c):.8g}'.replace('.','p').replace('-','m')
    return f'k_{k_graph}_c_{cslug}' if k_graph is not None else f'c_{cslug}'


def _run_signature(spec, method, phase, seed, c, temperature, median_distance, k_graph, ambient_bandwidth):
    return config_signature({
        'format_version':EXPERIMENT_FORMAT_VERSION,
        'phase':phase,'method':method,'spec':spec_metadata(spec),
        'training_config':asdict(TRAINING_CONFIG),'experiment_config':asdict(EXPERIMENT_CONFIG),
        'seed':int(seed),'c':float(c),'temperature':float(temperature),'median_distance':float(median_distance),
        'k_graph':None if k_graph is None else int(k_graph),'ambient_eval_bandwidth':float(ambient_bandwidth),
        'loss':'common_ambient_fixed_point_squared_norm',
    })


def save_trained_run(path, payload, model):
    saved=dict(payload)
    saved['model_state']=cpu_state_dict(model)
    atomic_torch_save(saved,path)


def load_trained_run(path, signature, spec):
    saved=maybe_load_run(path,signature,resume=RESUME,force=FORCE_RERUN)
    if saved is None: return None
    model=MLPDrift(TRAINING_CONFIG.latent_dim,TRAINING_CONFIG.hidden_dim,spec.manifold.ambient_dim).to(DEVICE)
    model.load_state_dict(saved.pop('model_state'))
    model.eval(); saved['model']=model
    return saved

## 14. Final test evaluation

For each of the three fixed test latent batches, the notebook records:

- `ambient_mmd_sq_raw`: raw generated samples versus real test data;
- `support_mean`, `support_median`, `support_p95`;
- `ambient_mmd_sq_projected`: after evaluation-only exact projection;
- `intrinsic_discrepancy_sq_projected`: exact geodesic-kernel discrepancy on the projected copy;
- `projection_gain_ambient_mmd_sq = raw - projected`.

For \(SO(3)\), raw orthogonality and determinant errors are also saved. Torus checkerboard retains tile accuracy.

In [ ]:
@torch.no_grad()
def evaluate_final_model(model, method, spec, ctx, test_data, test_noise_batches, ambient_bandwidth):
    rows=[]
    for z in test_noise_batches:
        raw=generate_raw_method(model,z,method,spec,ctx)
        projected=spec.manifold.project(raw)
        support=support_distance(raw,spec.manifold)
        raw_mmd=ambient_gaussian_mmd_sq(test_data,raw,ambient_bandwidth)
        projected_mmd=ambient_gaussian_mmd_sq(test_data,projected,ambient_bandwidth)
        row={
            'ambient_mmd_sq_raw':raw_mmd,
            'support_mean':float(support.mean()),
            'support_median':float(support.median()),
            'support_p95':float(torch.quantile(support,0.95)),
            'ambient_mmd_sq_projected':projected_mmd,
            'intrinsic_discrepancy_sq_projected':intrinsic_kernel_discrepancy_sq(test_data,projected,spec.manifold,bandwidth=1.0),
            'projection_gain_ambient_mmd_sq':raw_mmd-projected_mmd,
        }
        if spec.checkerboard_tiles is not None:
            row['tile_accuracy_projected']=checkerboard_accuracy(projected,spec.manifold,spec.checkerboard_tiles)
        if isinstance(spec.manifold,SO3):
            row.update(so3_constraint_diagnostics(raw))
        rows.append(row)

    summary={}
    for key in rows[0]:
        vals=[r[key] for r in rows]
        summary[key]={'mean':float(np.mean(vals)),'std_noise':float(np.std(vals,ddof=1) if len(vals)>1 else 0.0),'values':vals}
    return {'summary':summary,'individual_noise_batches':rows}


@torch.no_grad()
def estimate_real_vs_real_floor(spec, sample_size, ambient_bandwidth, repeats=None, seed=2026):
    repeats=EXPERIMENT_CONFIG.real_vs_real_repeats if repeats is None else repeats
    pool=spec.test_data
    g=torch.Generator(device='cpu').manual_seed(seed)
    rows=[]
    for _ in range(repeats):
        ia=torch.randint(0,len(pool),(sample_size,),generator=g)
        ib=torch.randint(0,len(pool),(sample_size,),generator=g)
        a=pool.index_select(0,ia).to(DEVICE)
        b=pool.index_select(0,ib).to(DEVICE)
        rows.append({
            'ambient_mmd_sq':ambient_gaussian_mmd_sq(a,b,ambient_bandwidth),
            'intrinsic_discrepancy_sq':intrinsic_kernel_discrepancy_sq(a,b,spec.manifold,bandwidth=1.0),
        })
    return {
        key:{'mean':float(np.mean([r[key] for r in rows])),'std':float(np.std([r[key] for r in rows],ddof=1) if repeats>1 else 0.0)}
        for key in rows[0]
    }

## 15. Fully retuned Exact-geodesic experiment

In [ ]:
# ============================================================
# EXACT GEODESIC — NEW RETUNED PROTOCOL
# ============================================================

GEODESIC_PROTOCOL_VERSION = 1
GEODESIC_METHOD = "exact"

# Keep this separate from every previous Exact experiment.
GEODESIC_EXPERIMENT_DIRNAME = "geodesic_loss_retuned"
GEODESIC_TUNING_DIRNAME = "tuning"
GEODESIC_FINAL_DIRNAME = "final_v4_50k_cosine_geodesic_loss"

FINAL_GEODESIC_STEPS = 50_000
FINAL_GEODESIC_LR_DECAY_START = int(TRAINING_CONFIG.lr_decay_start_step)
FINAL_GEODESIC_LR_SCHEDULE_END = FINAL_GEODESIC_STEPS

assert tuple(EXPERIMENT_CONFIG.tuning_seeds) == (43, 44, 45)
assert tuple(EXPERIMENT_CONFIG.final_seeds) == (101, 102, 103, 104, 105)
assert tuple(EXPERIMENT_CONFIG.c_grid) == (1/32, 1/16, 1/8, 1/4, 1/2, 1.0)
assert TRAINING_CONFIG.max_steps == 15_000
assert TRAINING_CONFIG.lr_decay_start_step == 5_000
assert FINAL_GEODESIC_STEPS == 50_000

def geodesic_root(spec):
    return method_dir(spec, "exact") / GEODESIC_EXPERIMENT_DIRNAME

def geodesic_tuning_dir(spec):
    return geodesic_root(spec) / GEODESIC_TUNING_DIRNAME

def geodesic_final_dir(spec):
    # Flat under <dataset>/exact/ so analysis/04_combine_and_report.ipynb
    # finds it. Tuning stays nested under geodesic_loss_retuned/ so it can
    # never collide with the ambient-loss Exact tuning from notebook 01.
    return method_dir(spec, "exact") / GEODESIC_FINAL_DIRNAME

print("Exact geodesic-loss RETUNED protocol")
print("-------------------------------------")
print("Tuning seeds:", EXPERIMENT_CONFIG.tuning_seeds)
print("Tuning c grid:", EXPERIMENT_CONFIG.c_grid)
print(f"Tuning max steps: {TRAINING_CONFIG.max_steps:,} (original early stopping ON)")
print(
    f"Tuning LR: {TRAINING_CONFIG.learning_rate:g} through step "
    f"{TRAINING_CONFIG.lr_decay_start_step:,}, cosine to "
    f"{TRAINING_CONFIG.min_learning_rate:g} at step {TRAINING_CONFIG.max_steps:,}"
)
print("Final seeds:", EXPERIMENT_CONFIG.final_seeds)
print(f"Final steps: {FINAL_GEODESIC_STEPS:,} (early stopping OFF)")
print(
    f"Final LR: {TRAINING_CONFIG.learning_rate:g} through step "
    f"{FINAL_GEODESIC_LR_DECAY_START:,}, cosine to "
    f"{TRAINING_CONFIG.min_learning_rate:g} at step {FINAL_GEODESIC_LR_SCHEDULE_END:,}"
)
print("Selection criterion: validation raw ambient Gaussian MMD²")
print("Tuning path: exact/geodesic_loss_retuned/tuning/")
print("Final path:  exact/" + GEODESIC_FINAL_DIRNAME + "/")


Exact geodesic-loss RETUNED protocol
-------------------------------------
Tuning seeds: (43, 44, 45)
Tuning c grid: (0.03125, 0.0625, 0.125, 0.25, 0.5, 1.0)
Tuning max steps: 15,000 (original early stopping ON)
Tuning LR: 0.001 through step 5,000, cosine to 1e-05 at step 15,000
Final seeds: (101, 102, 103, 104, 105)
Final steps: 50,000 (early stopping OFF)
Final LR: 0.001 through step 5,000, cosine to 1e-05 at step 50,000
Selection criterion: validation raw ambient Gaussian MMD²
Output relative path: exact/geodesic_loss_retuned/


In [ ]:
# ============================================================
# Geodesic fixed-point loss + cache signatures
# ============================================================

def geodesic_fixed_point_loss(gen, target, manifold):
    """
    L_geo = E[d_M(gen, stopgrad(target))^2].

    `target` is already constructed under torch.no_grad() in the trainer,
    but detach() is kept explicitly to preserve stop-gradient semantics.
    """
    target_frozen = target.detach()
    d = manifold.dist(gen, target_frozen)

    if not torch.isfinite(d).all():
        raise FloatingPointError("Non-finite geodesic distance encountered in loss.")

    return d.square().mean()


def _geodesic_run_signature(
    spec,
    phase,
    seed,
    c,
    temperature,
    median_distance,
    ambient_bandwidth,
):
    if phase == "tuning_geodesic_retuned":
        optimisation = {
            "max_steps": int(TRAINING_CONFIG.max_steps),
            "early_stopping": True,
            "lr_schedule_end": int(TRAINING_CONFIG.max_steps),
        }
    elif phase == "final_geodesic_retuned_50k_cosine":
        optimisation = {
            "max_steps": int(FINAL_GEODESIC_STEPS),
            "early_stopping": False,
            "lr_schedule_end": int(FINAL_GEODESIC_LR_SCHEDULE_END),
        }
    else:
        raise ValueError(phase)

    return config_signature({
        "format_version": EXPERIMENT_FORMAT_VERSION,
        "geodesic_protocol_version": GEODESIC_PROTOCOL_VERSION,
        "phase": phase,
        "method": "exact",
        "spec": spec_metadata(spec),
        "training_config": asdict(TRAINING_CONFIG),
        "experiment_config": asdict(EXPERIMENT_CONFIG),
        "optimisation": optimisation,
        "seed": int(seed),
        "c": float(c),
        "temperature": float(temperature),
        "median_distance": float(median_distance),
        "k_graph": None,
        "ambient_eval_bandwidth": float(ambient_bandwidth),
        "loss": "exact_squared_geodesic_fixed_point_stop_gradient",
        "selection_metric": "validation_raw_ambient_gaussian_mmd_sq",
    })


In [ ]:
# ============================================================
# Exact-geodesic trainer
#
# Tuning mode:
#   - max 15k
#   - original early stopping
#   - cosine reaches min LR at 15k
#
# Final mode:
#   - exactly 50k
#   - NO early stopping
#   - cosine reaches min LR at 50k
# ============================================================

def train_one_run_exact_geodesic(
    spec,
    temperature,
    seed,
    val_data,
    val_noise_batches,
    ambient_bandwidth,
    *,
    final_50k=False,
):
    cfg = TRAINING_CONFIG
    method = "exact"

    max_steps = FINAL_GEODESIC_STEPS if final_50k else cfg.max_steps
    lr_schedule_end = FINAL_GEODESIC_LR_SCHEDULE_END if final_50k else cfg.max_steps
    early_stopping_enabled = not final_50k

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = MLPDrift(
        cfg.latent_dim,
        cfg.hidden_dim,
        spec.manifold.ambient_dim,
    ).to(DEVICE)

    ema = copy.deepcopy(model).to(DEVICE).eval()
    for p in ema.parameters():
        p.requires_grad_(False)

    opt = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)

    ng = torch.Generator(device="cpu").manual_seed(seed + 10)
    dg = torch.Generator(device="cpu").manual_seed(seed + 20)

    best_score = float("inf")
    best_state = None
    best_step = None

    patience_ref = float("inf")
    bad = 0
    stopped_early = False
    stop_step = max_steps

    history = {
        "step": [],
        "loss": [],
        "smoothed_loss": [],
        "gradient_norm": [],
        "cap_fraction": [],
        "graph_finite_fraction": [],
        "support_mean": [],
        "learning_rate": [],
        "validation": [],
    }

    smoothed = None
    batch = spec.effective_batch_size

    phase_label = "FINAL 50K" if final_50k else "TUNING"

    pbar = tqdm(
        range(1, max_steps + 1),
        desc=(
            f"GEO {phase_label} | {spec.name} | Exact | "
            f"seed={seed} | tau={temperature:.4g}"
        ),
    )

    for step in pbar:
        lr = cosine_learning_rate(
            step,
            cfg.learning_rate,
            cfg.min_learning_rate,
            cfg.lr_decay_start_step,
            lr_schedule_end,
        )
        for group in opt.param_groups:
            group["lr"] = lr

        pos = spec.train_sampler(batch, generator=dg).to(DEVICE)
        z = torch.randn(
            batch,
            cfg.latent_dim,
            generator=ng,
        ).to(DEVICE)

        raw = model(z)
        gen = initial_map(raw, method, spec, None)

        with torch.no_grad():
            V, finite_fraction = compute_method_drift(
                gen,
                pos,
                method,
                spec,
                temperature,
                None,
            )

            if not torch.isfinite(V).all():
                raise FloatingPointError(
                    f"Non-finite drift: {spec.name}/exact/seed={seed}/step={step}"
                )

            V, cap_fraction = cap_method_step(
                V,
                method,
                spec,
                cfg.max_step,
            )

            target = construct_target(
                gen,
                V,
                method,
                spec,
                None,
            )

        # ONLY intended training-objective change.
        loss = geodesic_fixed_point_loss(
            gen,
            target,
            spec.manifold,
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite loss: {spec.name}/exact/seed={seed}/step={step}"
            )

        opt.zero_grad(set_to_none=True)
        loss.backward()

        # Same gradient clipping as the locked experiment.
        grad_norm = float(
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg.gradient_clip_norm,
            )
        )

        opt.step()
        update_ema_model(ema, model, cfg.ema_decay)

        loss_value = float(loss.item())
        smoothed = (
            loss_value
            if smoothed is None
            else 0.96 * smoothed + 0.04 * loss_value
        )

        if step == 1 or step % cfg.history_every == 0:
            with torch.no_grad():
                support_mean = float(
                    support_distance(gen, spec.manifold).mean()
                )

            history["step"].append(step)
            history["loss"].append(loss_value)
            history["smoothed_loss"].append(smoothed)
            history["gradient_norm"].append(grad_norm)
            history["cap_fraction"].append(cap_fraction)
            history["graph_finite_fraction"].append(finite_fraction)
            history["support_mean"].append(support_mean)
            history["learning_rate"].append(lr)

        if step % cfg.validate_every == 0 or step == max_steps:
            # Selection metric is deliberately unchanged.
            score, score_std = validation_score(
                ema,
                method,
                spec,
                None,
                val_data,
                val_noise_batches,
                ambient_bandwidth,
            )

            history["validation"].append({
                "step": int(step),
                "raw_ambient_mmd_sq": float(score),
                "std_noise": float(score_std),
            })

            if score < best_score:
                best_score = score
                best_step = step
                best_state = copy.deepcopy(ema.state_dict())

            # EXACTLY the old tuning early-stopping rule.
            if early_stopping_enabled and step >= cfg.early_stopping_start_step:
                if _meaningful_improvement(
                    score,
                    patience_ref,
                    cfg.min_delta_rel,
                    cfg.min_delta_abs,
                ):
                    patience_ref = score
                    bad = 0
                else:
                    bad += 1

                if bad >= cfg.early_stopping_patience:
                    stopped_early = True
                    stop_step = step
                    break

        pbar.set_postfix(
            loss=f"{smoothed:.2e}",
            cap=f"{cap_fraction:.0%}",
            lr=f"{lr:.1e}",
            best_val=(
                f"{best_score:.2e}"
                if math.isfinite(best_score)
                else "—"
            ),
        )

    if best_state is None:
        raise RuntimeError("No validation checkpoint was recorded.")

    ema.load_state_dict(best_state)
    ema.eval()

    return {
        "model": ema,
        "best_validation_score": float(best_score),
        "best_step": int(best_step),
        "stop_step": int(stop_step),
        "stopped_early": bool(stopped_early),
        "history": history,
    }


## 16. Fresh geodesic-loss hyperparameter tuning

In [ ]:
# ============================================================
# FULL Exact-geodesic hyperparameter selection
#
# SAME selection protocol as original Exact:
#   c grid = 1/32,...,1
#   tuning seeds = 43,44,45
#   validation criterion = mean raw ambient MMD²
#
# DIFFERENCE:
#   every tuning model is now actually TRAINED with geodesic loss.
# ============================================================

def tune_exact_geodesic(spec, scales, val_data, val_noise_batches):
    out_dir = geodesic_tuning_dir(spec)
    out_dir.mkdir(parents=True, exist_ok=True)

    median = float(scales["exact_median_distance"])
    ambient_bw = float(scales["ambient_eval_bandwidth"])

    rows = []

    for c in EXPERIMENT_CONFIG.c_grid:
        c = float(c)
        temperature = float(c * median)

        for seed in EXPERIMENT_CONFIG.tuning_seeds:
            seed = int(seed)

            slug = _candidate_slug(None, c)
            run_path = out_dir / slug / f"seed_{seed}.pt"

            sig = _geodesic_run_signature(
                spec,
                "tuning_geodesic_retuned",
                seed,
                c,
                temperature,
                median,
                ambient_bw,
            )

            saved = load_trained_run(
                run_path,
                sig,
                spec,
            )

            if saved is None:
                trained = train_one_run_exact_geodesic(
                    spec,
                    temperature,
                    seed,
                    val_data,
                    val_noise_batches,
                    ambient_bw,
                    final_50k=False,
                )

                payload = {
                    "run_signature": sig,
                    "phase": "tuning_geodesic_retuned",
                    "method": "exact",
                    "loss": "squared_geodesic_fixed_point",
                    "seed": seed,
                    "k_graph": None,
                    "c": c,
                    "temperature": temperature,
                    "median_distance": median,
                    "best_validation_score": trained["best_validation_score"],
                    "best_step": trained["best_step"],
                    "stop_step": trained["stop_step"],
                    "stopped_early": trained["stopped_early"],
                    "history": trained["history"],
                }

                save_trained_run(
                    run_path,
                    payload,
                    trained["model"],
                )

                saved = {
                    **payload,
                    "model": trained["model"],
                }

            rows.append({
                "manifold": spec.manifold_kind,
                "dataset": spec.name,
                "method": "exact_geodesic_retuned",
                "c": c,
                "seed": seed,
                "temperature": temperature,
                "median_distance": median,
                "best_validation_score": float(saved["best_validation_score"]),
                "best_step": int(saved["best_step"]),
                "stop_step": int(saved["stop_step"]),
                "stopped_early": bool(saved["stopped_early"]),
            })

            if "model" in saved:
                del saved["model"]

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    runs = pd.DataFrame(rows)

    summary = (
        runs
        .groupby(["c"], as_index=False)
        .agg(
            mean_validation_score=("best_validation_score", "mean"),
            std_validation_score=("best_validation_score", "std"),
            mean_best_step=("best_step", "mean"),
            mean_stop_step=("stop_step", "mean"),
            early_stop_fraction=("stopped_early", "mean"),
            median_distance=("median_distance", "first"),
            temperature=("temperature", "first"),
        )
        .sort_values("mean_validation_score", ascending=True)
        .reset_index(drop=True)
    )

    winner = summary.iloc[0].to_dict()

    selected_c = float(winner["c"])
    selected_temperature = float(winner["temperature"])
    selected_median = float(winner["median_distance"])

    boundary_c = selected_c in (
        min(EXPERIMENT_CONFIG.c_grid),
        max(EXPERIMENT_CONFIG.c_grid),
    )

    result = {
        "method": "exact",
        "variant": "geodesic_loss_retuned",
        "selected_k_graph": None,
        "selected_c": selected_c,
        "selected_temperature": selected_temperature,
        "selected_median_distance": selected_median,
        "selected_mean_validation_score": float(
            winner["mean_validation_score"]
        ),
        "selected_std_validation_score": (
            float(winner["std_validation_score"])
            if math.isfinite(float(winner["std_validation_score"]))
            else 0.0
        ),
        "winner_on_c_boundary": bool(boundary_c),
        "loss": "squared_geodesic_fixed_point",
        "selection_metric": "raw_ambient_gaussian_mmd_sq",
        "tuning_seeds": list(EXPERIMENT_CONFIG.tuning_seeds),
        "runs": runs,
        "summary_table": summary,
    }

    runs.to_csv(
        out_dir / "tuning_runs.csv",
        index=False,
    )
    summary.to_csv(
        out_dir / "tuning_summary.csv",
        index=False,
    )

    atomic_json_save(
        {
            k: v
            for k, v in result.items()
            if k not in {"runs", "summary_table"}
        },
        out_dir / "selected_hyperparameters.json",
    )

    print("\nSelected geodesic-loss Exact hyperparameter")
    print("-------------------------------------------")
    print(f"Dataset: {spec.manifold_kind}/{spec.name}")
    print(f"selected c: {selected_c:g}")
    print(f"selected temperature: {selected_temperature:.8g}")
    print(
        "validation MMD²: "
        f"{result['selected_mean_validation_score']:.6g} "
        f"± {result['selected_std_validation_score']:.3g}"
    )
    print(f"winner on c-grid boundary: {boundary_c}")

    display(summary)

    return result


## 17. 50,000-step final fitting

In [ ]:
# ============================================================
# 50,000-step final fitting using the NEWLY RETUNED c
#
# - five fresh final seeds
# - no early stopping
# - lr=1e-3 through 5k, cosine -> 1e-5 at 50k
# - checkpoint selection ONLY by validation raw ambient MMD²
# - immediate frozen-test evaluation after each completed seed
# ============================================================

def _seed_test_json_payload(spec, selection, saved_run):
    return {
        "protocol": "exact_geodesic_loss_retuned_50k_cosine",
        "manifold": spec.manifold_kind,
        "dataset": spec.name,
        "method": "exact",
        "method_label": "Exact (geodesic loss, retuned)",
        "seed": int(saved_run["seed"]),
        "selected_c": float(selection["selected_c"]),
        "selected_temperature": float(selection["selected_temperature"]),
        "selected_median_distance": float(selection["selected_median_distance"]),
        "_best_validation_step": int(saved_run["best_step"]),
        "_best_validation_score": float(saved_run["best_validation_score"]),
        "summary": saved_run["test_evaluation"]["summary"],
        "individual_noise_batches": saved_run["test_evaluation"]["individual_noise_batches"],
    }


def run_final_exact_geodesic(
    spec,
    selection,
    scales,
    val_data,
    test_data,
    val_noise_batches,
    test_noise_batches,
):
    out_dir = geodesic_final_dir(spec)
    out_dir.mkdir(parents=True, exist_ok=True)

    c = float(selection["selected_c"])
    median = float(scales["exact_median_distance"])
    temperature = float(c * median)
    ambient_bw = float(scales["ambient_eval_bandwidth"])

    # Audit against what was actually selected during geodesic tuning.
    if not math.isclose(
        median,
        float(selection["selected_median_distance"]),
        rel_tol=1e-10,
        abs_tol=1e-12,
    ):
        raise RuntimeError(
            "Exact median distance changed between tuning and final fitting."
        )

    runs = []

    for seed in EXPERIMENT_CONFIG.final_seeds:
        seed = int(seed)

        run_path = out_dir / f"seed_{seed}.pt"
        test_json_path = out_dir / f"seed_{seed}_test_evaluation.json"

        sig = _geodesic_run_signature(
            spec,
            "final_geodesic_retuned_50k_cosine",
            seed,
            c,
            temperature,
            median,
            ambient_bw,
        )

        saved = load_trained_run(
            run_path,
            sig,
            spec,
        )

        if saved is None:
            trained = train_one_run_exact_geodesic(
                spec,
                temperature,
                seed,
                val_data,
                val_noise_batches,
                ambient_bw,
                final_50k=True,
            )

            # Test set is touched only now, after validation checkpoint selection.
            evaluation = evaluate_final_model(
                trained["model"],
                "exact",
                spec,
                None,
                test_data,
                test_noise_batches,
                ambient_bw,
            )

            payload = {
                "run_signature": sig,
                "phase": "final_geodesic_retuned_50k_cosine",
                "method": "exact",
                "variant": "geodesic_loss_retuned",
                "loss": "squared_geodesic_fixed_point",
                "seed": seed,
                "k_graph": None,
                "c": c,
                "temperature": temperature,
                "median_distance": median,
                "best_validation_score": trained["best_validation_score"],
                "best_step": trained["best_step"],
                "stop_step": FINAL_GEODESIC_STEPS,
                "stopped_early": False,
                "history": trained["history"],
                "test_evaluation": evaluation,
            }

            save_trained_run(
                run_path,
                payload,
                trained["model"],
            )

            saved = {
                **payload,
                "model": trained["model"],
            }

            # Immediate per-seed JSON makes interruption/resume safe and
            # compatible with the style of the previous 50k experiment.
            atomic_json_save(
                _seed_test_json_payload(spec, selection, saved),
                test_json_path,
            )

        else:
            # Recreate test JSON if checkpoint exists but JSON is missing.
            if "test_evaluation" not in saved:
                evaluation = evaluate_final_model(
                    saved["model"],
                    "exact",
                    spec,
                    None,
                    test_data,
                    test_noise_batches,
                    ambient_bw,
                )
                saved["test_evaluation"] = evaluation

                payload = {
                    k: v
                    for k, v in saved.items()
                    if k != "model"
                }
                save_trained_run(
                    run_path,
                    payload,
                    saved["model"],
                )

            if not test_json_path.exists():
                atomic_json_save(
                    _seed_test_json_payload(spec, selection, saved),
                    test_json_path,
                )

        runs.append({
            k: v
            for k, v in saved.items()
            if k != "model"
        })

        print(
            f"[DONE] {spec.name} seed={seed} | "
            f"best step={saved['best_step']:,} | "
            f"val={saved['best_validation_score']:.6g} | "
            f"test raw MMD²="
            f"{saved['test_evaluation']['summary']['ambient_mmd_sq_raw']['mean']:.6g}"
        )

        if "model" in saved:
            del saved["model"]

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metric_names = sorted(
        runs[0]["test_evaluation"]["summary"].keys()
    )

    summary = {}

    for metric in metric_names:
        values = [
            float(
                r["test_evaluation"]["summary"][metric]["mean"]
            )
            for r in runs
        ]

        summary[metric] = {
            "mean_over_training_seeds": float(np.mean(values)),
            "std_over_training_seeds": float(
                np.std(values, ddof=1)
                if len(values) > 1
                else 0.0
            ),
            "values_by_seed": {
                str(int(r["seed"])): float(v)
                for r, v in zip(runs, values)
            },
        }

    result = {
        "method": "exact",
        "variant": "geodesic_loss_retuned",
        "selected_k_graph": None,
        "selected_c": c,
        "selected_temperature": temperature,
        "median_distance": median,
        "runs": runs,
        "summary": summary,
    }

    atomic_json_save(
        {
            "method": "exact",
            "variant": "geodesic_loss_retuned",
            "selected_c": c,
            "selected_temperature": temperature,
            "median_distance": median,
            "summary": summary,
        },
        out_dir / "summary.json",
    )

    return result


## 18. Run the experiment

In [ ]:
# ============================================================
# DATASET RUNNER
#
# This performs, for each selected dataset:
#   1. fresh geodesic-loss c tuning;
#   2. selection by validation raw ambient MMD²;
#   3. five 50k final runs using that selected c;
#   4. frozen-test evaluation and atomic saving.
#
# RESUME=True means completed tuning/final seeds are reused.
# ============================================================

def resolve_geodesic_active_datasets():
    names = list(DATASETS)

    if ACTIVE_DATASETS is not None:
        requested = list(ACTIVE_DATASETS)
        missing = [x for x in requested if x not in DATASETS]
        if missing:
            raise KeyError(
                f"Unknown ACTIVE_DATASETS: {missing}"
            )
        names = requested

    return names


def run_exact_geodesic_retuned_dataset(spec):
    print("\n" + "=" * 104)
    print(
        f"{spec.manifold_kind.upper()} / {spec.name} / "
        "EXACT — GEODESIC LOSS, FRESHLY RETUNED"
    )
    print("=" * 104)

    out_root = geodesic_root(spec)
    out_root.mkdir(parents=True, exist_ok=True)

    val_data, test_data, val_noise, test_noise = prepare_eval_objects(spec)
    scales = compute_dataset_scales(spec)

    # Save a self-contained metadata record for this new variant.
    atomic_json_save(
        {
            **spec_metadata(spec),
            **scales,
            "variant": "exact_geodesic_loss_retuned",
            "loss": "squared_geodesic_fixed_point",
            "selection_metric": "raw_ambient_gaussian_mmd_sq",
            "tuning_seeds": list(EXPERIMENT_CONFIG.tuning_seeds),
            "final_seeds": list(EXPERIMENT_CONFIG.final_seeds),
            "c_grid": list(EXPERIMENT_CONFIG.c_grid),
            "tuning_max_steps": int(TRAINING_CONFIG.max_steps),
            "final_steps": int(FINAL_GEODESIC_STEPS),
            "final_lr_decay_start": int(FINAL_GEODESIC_LR_DECAY_START),
            "final_lr_schedule_end": int(FINAL_GEODESIC_LR_SCHEDULE_END),
        },
        out_root / "experiment_metadata.json",
    )

    tuning = tune_exact_geodesic(
        spec,
        scales,
        val_data,
        val_noise,
    )

    final = run_final_exact_geodesic(
        spec,
        tuning,
        scales,
        val_data,
        test_data,
        val_noise,
        test_noise,
    )

    raw = final["summary"]["ambient_mmd_sq_raw"]

    print("\nFINAL RETUNED GEODESIC EXACT RESULT")
    print(
        f"{spec.manifold_kind}/{spec.name}: "
        f"raw MMD² = {raw['mean_over_training_seeds']:.6g} "
        f"± {raw['std_over_training_seeds']:.3g}"
    )

    return {
        "spec": spec,
        "scales": scales,
        "tuning": tuning,
        "final": final,
    }


GEODESIC_RESULTS = {}

for dataset_name in resolve_geodesic_active_datasets():
    GEODESIC_RESULTS[dataset_name] = (
        run_exact_geodesic_retuned_dataset(
            DATASETS[dataset_name]
        )
    )

print("\nCompleted retuned geodesic-loss datasets:")
print(list(GEODESIC_RESULTS))



SPHERE / sphere_spiral / EXACT — GEODESIC LOSS, FRESHLY RETUNED
Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/sphere_spiral/exact/geodesic_loss_retuned/tuning/c_0p03125/seed_43.pt
Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/sphere_spiral/exact/geodesic_loss_retuned/tuning/c_0p03125/seed_44.pt
Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/sphere_spiral/exact/geodesic_loss_retuned/tuning/c_0p03125/seed_45.pt
Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/sphere_spiral/exact/geodesic_loss_retuned/tuning/c_0p0625/seed_43.pt
Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/sphere_spiral/exact/geodesic_loss_retuned/tuning/c_0p0625/seed_44.pt
Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/sphere_spiral/exact/geodesic_loss_r

,c,mean_validation_score,std_validation_score,mean_best_step,mean_stop_step,early_stop_fraction,median_distance,temperature
0,0.12500,0.000225,0.000065,10166.666667,14000.000000,0.333333,1.580098,0.197512
1,0.50000,0.000249,0.000051,7000.000000,12000.000000,1.000000,1.580098,0.790049
2,0.06250,0.000280,0.000062,6500.000000,11500.000000,1.000000,1.580098,0.098756
3,0.25000,0.000306,0.000090,7333.333333,12000.000000,1.000000,1.580098,0.395024
4,0.03125,0.000317,0.000180,8166.666667,12500.000000,0.666667,1.580098,0.049378
5,1.00000,0.159838,0.012411,500.000000,11166.666667,1.000000,1.580098,1.580098


Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/sphere_spiral/exact/geodesic_loss_retuned/final_50k_cosine/seed_101.pt
[DONE] sphere_spiral seed=101 | best step=49,000 | val=0.000107586 | test raw MMD²=0.000359118
Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/sphere_spiral/exact/geodesic_loss_retuned/final_50k_cosine/seed_102.pt
[DONE] sphere_spiral seed=102 | best step=12,000 | val=0.000228643 | test raw MMD²=0.000348111
Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/sphere_spiral/exact/geodesic_loss_retuned/final_50k_cosine/seed_103.pt
[DONE] sphere_spiral seed=103 | best step=31,500 | val=0.000186066 | test raw MMD²=0.000534058
Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/sphere_spiral/exact/geodesic_loss_retuned/final_50k_cosine/seed_104.pt
[DONE] sphere_spiral seed=104 | best step=23,500 | val=0.000213

,c,mean_validation_score,std_validation_score,mean_best_step,mean_stop_step,early_stop_fraction,median_distance,temperature
0,0.03125,0.003453,0.000751,7666.666667,12000.000000,0.666667,1.364063,0.042627
1,0.12500,0.004268,0.001243,7166.666667,14500.000000,0.666667,1.364063,0.170508
2,0.06250,0.004540,0.001812,7666.666667,13833.333333,0.333333,1.364063,0.085254
3,0.50000,0.006814,0.003154,9500.000000,13833.333333,0.666667,1.364063,0.682031
4,0.25000,0.008961,0.003125,13333.333333,13833.333333,0.333333,1.364063,0.341016
5,1.00000,0.050286,0.016523,500.000000,12666.666667,0.666667,1.364063,1.364063


Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/volcano/exact/geodesic_loss_retuned/final_50k_cosine/seed_101.pt
[DONE] volcano seed=101 | best step=10,500 | val=0.00356919 | test raw MMD²=0.00779865
Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/volcano/exact/geodesic_loss_retuned/final_50k_cosine/seed_102.pt
[DONE] volcano seed=102 | best step=3,000 | val=0.00457046 | test raw MMD²=0.0132378
Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/volcano/exact/geodesic_loss_retuned/final_50k_cosine/seed_103.pt
[DONE] volcano seed=103 | best step=5,000 | val=0.00452252 | test raw MMD²=0.00876703
Resuming from: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/sphere/volcano/exact/geodesic_loss_retuned/final_50k_cosine/seed_104.pt
[DONE] volcano seed=104 | best step=6,000 | val=0.00295997 | test raw MMD²=0.0102584
Resuming from: /content/drive

...

[DONE] CMU seed=103 | best step=500 | val=0.001516 | test raw MMD²=0.00506153


GEO FINAL 50K | CMU | Exact | seed=104 | tau=0.04664:   0%|          | 0/50000 [00:00<?, ?it/s]

[DONE] CMU seed=104 | best step=4,500 | val=0.00147198 | test raw MMD²=0.00190117


GEO FINAL 50K | CMU | Exact | seed=105 | tau=0.04664:   0%|          | 0/50000 [00:00<?, ?it/s]

[DONE] CMU seed=105 | best step=13,000 | val=0.00129286 | test raw MMD²=0.00329053

FINAL RETUNED GEODESIC EXACT RESULT
so3/CMU: raw MMD² = 0.00317441 ± 0.00121

Completed retuned geodesic-loss datasets:
['sphere_spiral', 'volcano', 'earthquake', 'fire', 'flood', 'torus_checkerboard', 'general', 'glycine', 'proline', 'pre_pro', 'so3_synthetic', 'BMLhandball', 'DanceDB', 'BMLmovi', 'CMU']


## 19. Progress / result monitor

In [ ]:
# ============================================================
# CURRENT RESULTS / COMPLETION MONITOR
#
# Safe to rerun. Reads only the NEW geodesic-retuned folders.
# ============================================================

monitor_rows = []

for dataset_name, spec in DATASETS.items():
    tsel = (
        geodesic_tuning_dir(spec)
        / "selected_hyperparameters.json"
    )
    fdir = geodesic_final_dir(spec)

    selected = None
    if tsel.exists():
        with open(tsel, "r") as f:
            selected = json.load(f)

    completed_seeds = []
    seed_raw_mmd = {}

    for seed in EXPERIMENT_CONFIG.final_seeds:
        p = fdir / f"seed_{seed}_test_evaluation.json"

        if p.exists():
            with open(p, "r") as f:
                e = json.load(f)

            completed_seeds.append(int(seed))
            seed_raw_mmd[str(seed)] = float(
                e["summary"]["ambient_mmd_sq_raw"]["mean"]
            )

    vals = list(seed_raw_mmd.values())

    monitor_rows.append({
        "manifold": spec.manifold_kind,
        "dataset": dataset_name,
        "tuning_complete": selected is not None,
        "selected_c": (
            np.nan
            if selected is None
            else float(selected["selected_c"])
        ),
        "selected_temperature": (
            np.nan
            if selected is None
            else float(selected["selected_temperature"])
        ),
        "final_seeds_complete": len(completed_seeds),
        "completed_seeds": completed_seeds,
        "raw_mmd_mean_so_far": (
            np.nan
            if not vals
            else float(np.mean(vals))
        ),
        "raw_mmd_sd_so_far": (
            np.nan
            if len(vals) < 2
            else float(np.std(vals, ddof=1))
        ),
    })

GEODESIC_MONITOR = pd.DataFrame(monitor_rows)

display(GEODESIC_MONITOR)

monitor_csv = (
    RESULTS_ROOT
    / "exact_geodesic_retuned_completion_monitor.csv"
)
GEODESIC_MONITOR.to_csv(monitor_csv, index=False)

print("Saved monitor:", monitor_csv)


,manifold,dataset,tuning_complete,selected_c,selected_temperature,final_seeds_complete,completed_seeds,raw_mmd_mean_so_far,raw_mmd_sd_so_far
0,sphere,sphere_spiral,True,0.12500,0.197512,5,"[101, 102, 103, 104, 105]",0.000498,0.000148
1,sphere,volcano,True,0.03125,0.042627,5,"[101, 102, 103, 104, 105]",0.010680,0.002537
2,sphere,earthquake,True,0.50000,0.679775,5,"[101, 102, 103, 104, 105]",0.001321,0.000428
3,sphere,fire,True,0.12500,0.160261,5,"[101, 102, 103, 104, 105]",0.000742,0.000153
4,sphere,flood,True,0.12500,0.166830,5,"[101, 102, 103, 104, 105]",0.004041,0.001654
5,torus,torus_checkerboard,True,0.06250,0.156281,5,"[101, 102, 103, 104, 105]",0.000892,0.000158
6,torus,general,True,0.25000,0.485047,5,"[101, 102, 103, 104, 105]",0.000555,0.000364
7,torus,glycine,True,0.03125,0.085190,5,"[101, 102, 103, 104, 105]",0.002588,0.000303
8,torus,proline,True,0.03125,0.043577,5,"[101, 102, 103, 104, 105]",0.000859,0.000263
9,torus,pre_pro,True,0.25000,0.268424,5,"[101, 102, 103, 104, 105]",0.006692,0.001055


Saved monitor: /content/drive/MyDrive/dissertation/final_four_model_all_manifolds_v1/exact_geodesic_retuned_completion_monitor.csv


## 20. Optional hyperparameter-selection comparison

In [ ]:
# ============================================================
# OPTIONAL: compare newly selected c against old ambient-loss Exact c
#
# This is diagnostic only and does not affect tuning/final fitting.
# ============================================================

comparison_rows = []

for dataset_name, spec in DATASETS.items():
    new_path = (
        geodesic_tuning_dir(spec)
        / "selected_hyperparameters.json"
    )

    old_path = (
        method_dir(spec, "exact")
        / "tuning"
        / "selected_hyperparameters.json"
    )

    if not new_path.exists():
        continue

    with open(new_path, "r") as f:
        new_sel = json.load(f)

    old_sel = None
    if old_path.exists():
        with open(old_path, "r") as f:
            old_sel = json.load(f)

    comparison_rows.append({
        "manifold": spec.manifold_kind,
        "dataset": dataset_name,
        "old_ambient_loss_selected_c": (
            np.nan
            if old_sel is None
            else float(old_sel["selected_c"])
        ),
        "new_geodesic_loss_selected_c": float(
            new_sel["selected_c"]
        ),
        "same_c": (
            False
            if old_sel is None
            else bool(
                math.isclose(
                    float(old_sel["selected_c"]),
                    float(new_sel["selected_c"]),
                )
            )
        ),
        "new_validation_mean": float(
            new_sel["selected_mean_validation_score"]
        ),
        "new_validation_sd": float(
            new_sel["selected_std_validation_score"]
        ),
        "new_winner_on_c_boundary": bool(
            new_sel["winner_on_c_boundary"]
        ),
    })

GEODESIC_SELECTION_COMPARISON = pd.DataFrame(
    comparison_rows
)

display(GEODESIC_SELECTION_COMPARISON)


,manifold,dataset,old_ambient_loss_selected_c,new_geodesic_loss_selected_c,same_c,new_validation_mean,new_validation_sd,new_winner_on_c_boundary
0,sphere,sphere_spiral,0.06250,0.12500,False,0.000225,0.000065,False
1,sphere,volcano,0.25000,0.03125,False,0.003453,0.000751,True
2,sphere,earthquake,0.06250,0.50000,False,0.000654,0.000072,False
3,sphere,fire,0.12500,0.12500,True,0.000295,0.000079,False
4,sphere,flood,0.12500,0.12500,True,0.001089,0.000247,False
5,torus,torus_checkerboard,0.06250,0.06250,True,0.000424,0.000159,False
6,torus,general,0.12500,0.25000,False,0.000669,0.000133,False
7,torus,glycine,0.06250,0.03125,False,0.000510,0.000127,True
8,torus,proline,0.12500,0.03125,False,0.000506,0.000132,True
9,torus,pre_pro,0.25000,0.25000,True,0.001760,0.000274,False


## Interpretation / safeguards

This notebook intentionally creates a **new model variant**, rather than silently replacing
the original Exact model.

The primary four-model experiment used a common ambient fixed-point loss to isolate the
effect of geometric operations. This notebook answers a different ablation question:

> If Exact is allowed to use an intrinsically natural geodesic regression loss, and that
> variant is **properly retuned under the same validation protocol**, how well does it perform?

Do not use the final test results to change the selected \(c\), extend the grid, or otherwise
modify the protocol. If a selected \(c\) lies on the grid boundary, report the boundary flag
as a sensitivity caveat rather than expanding the grid after seeing test performance.
